In [ ]:
import os
import snowflake.connector as sc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from dotenv import load_dotenv
load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.getcwd()), '.env'))

%config InlineBackend.figure_format = 'retina'

In [ ]:
conn_params = {
    'account': "HOFEOUE-CPB80774",
    'user': "GARRETT@ALIGNABLE.COM",
    'authenticator': "externalbrowser",
    'role': "DEVELOPMENT_ROLE",
    'warehouse': "DEV_SMALL",
    'database': "ALIGNABLE_REPORTING",
    'schema':"EVENT_STREAMS",
    'paramstyle': 'qmark'
}

alignable_reporting_conn = sc.connect(**conn_params)
reporting_cursor = alignable_reporting_conn.cursor()

In [ ]:
def query(query_str):
	return reporting_cursor.execute(query_str).fetch_pandas_all()

# Cancellation Rate by Customer Type

Business customer type resolved via `relationships_business_customer_metadata` (preferred) with fallback to `business_flags`.

In [ ]:
# Debug: inspect user_ids parsing
# user_ids format is {"2414167","16682199"} — Postgres array string
query("""
    SELECT
        c."user_ids",
        SPLIT_PART(TRANSLATE(c."user_ids", '{}"', ''), ',', 1) AS user1,
        SPLIT_PART(TRANSLATE(c."user_ids", '{}"', ''), ',', 2) AS user2,
        a."conversation_id",
        a."meeting_lifecycle"
    FROM OPENFLOW_DB."public"."conversation_meeting_analyses" a
    INNER JOIN OPENFLOW_DB."public"."conversations" c
        ON c."id" = a."conversation_id"
    LIMIT 10
""")

In [ ]:
from adjustText import adjust_text

BASE_QUERY = """
WITH conversation_users AS (
    SELECT
        a."conversation_id",
        a."meeting_lifecycle",
        SPLIT_PART(TRANSLATE(c."user_ids", '{{}}"', ''), ',', 1) AS user1,
        SPLIT_PART(TRANSLATE(c."user_ids", '{{}}"', ''), ',', 2) AS user2
    FROM OPENFLOW_DB."public"."conversation_meeting_analyses" a
    INNER JOIN OPENFLOW_DB."public"."conversations" c
        ON c."id" = a."conversation_id"
),
user_stage_counts AS (
    SELECT
        u.USER_ID,
        u.MEETING_LIFECYCLE,
        COUNT(*) AS meetings_at_stage
    FROM (
        SELECT user1 AS USER_ID, "meeting_lifecycle" AS MEETING_LIFECYCLE FROM conversation_users
        UNION ALL
        SELECT user2 AS USER_ID, "meeting_lifecycle" AS MEETING_LIFECYCLE FROM conversation_users
    ) u
    GROUP BY 1, 2
),
business_stage_counts AS (
    SELECT DISTINCT
        u.BUSINESS_ID,
        usc.MEETING_LIFECYCLE,
        usc.meetings_at_stage
    FROM ALIGNABLE_REPORTING_V2.CORE.DIM_USER u
    INNER JOIN user_stage_counts usc ON usc.USER_ID = u.ID::VARCHAR
    WHERE u.IS_PRIMARY_USER = TRUE
),
business_customer_type AS (
    SELECT
        b.BUSINESS_ID,
        CASE
            WHEN rcm."customer_type" = 1 THEN 'b2b'
            WHEN rcm."customer_type" = 0 THEN 'b2c'
            WHEN rcm."customer_type" = 2 THEN 'both'
            WHEN rcm."customer_type" = 3 THEN 'none'
            WHEN bf."customer_type" = 0 THEN 'b2b'
            WHEN bf."customer_type" = 1 THEN 'b2c'
            ELSE 'unknown'
        END AS business_type
    FROM business_stage_counts b
    LEFT JOIN OPENFLOW_DB."public"."relationships_business_customer_metadata" rcm
        ON rcm."business_id" = b.BUSINESS_ID
    LEFT JOIN OPENFLOW_DB."public"."business_flags" bf
        ON bf."business_id" = b.BUSINESS_ID
    QUALIFY ROW_NUMBER() OVER (PARTITION BY b.BUSINESS_ID ORDER BY rcm."customer_type" NULLS LAST) = 1
),
thresholds AS (
    SELECT column1 AS threshold FROM VALUES (1),(2),(3),(4),(5),(6),(7),(8),(9),(10)
),
cumulative AS (
    SELECT
        b.MEETING_LIFECYCLE,
        t.threshold,
        b.BUSINESS_ID
    FROM business_stage_counts b
    CROSS JOIN thresholds t
    INNER JOIN business_customer_type bct ON bct.BUSINESS_ID = b.BUSINESS_ID
    WHERE b.meetings_at_stage >= t.threshold
      AND bct.business_type IN ({filter})
)
SELECT
    CASE c.MEETING_LIFECYCLE
        WHEN 6 THEN 'completed (6)'
        WHEN 5 THEN 'scheduled (5)'
        WHEN 4 THEN 'mutual_intent (4)'
    END AS lifecycle_stage,
    c.MEETING_LIFECYCLE AS lifecycle_sort,
    c.threshold,
    COUNT_IF(s.SUBSCRIPTION_STATUS = 'churned') AS churned,
    COUNT_IF(s.SUBSCRIPTION_STATUS = 'active') AS active,
    churned + active AS total,
    ROUND(churned / NULLIF(churned + active, 0) * 100, 2) AS cancellation_rate_pct
FROM ALIGNABLE_REPORTING_V2.MEMBERSHIPS.DIM_MEMBERSHIP_SUBSCRIPTIONS s
INNER JOIN cumulative c ON c.BUSINESS_ID = s.BUSINESS_ID
WHERE c.MEETING_LIFECYCLE IN (4, 5, 6)
  AND s.ACTIVE_AT >= GREATEST('2026-01-20'::DATE, DATEADD(DAY, -90, CURRENT_DATE()))
GROUP BY 1, 2, 3
ORDER BY lifecycle_sort DESC, threshold ASC
"""

lifecycle_order = [
    'completed (6)', 'scheduled (5)', 'mutual_intent (4)'
]

colors = {
    'completed (6)': '#2ecc71',
    'scheduled (5)': '#3498db',
    'mutual_intent (4)': '#9b59b6',
}

linestyles = {}

def plot_cancellation_rate(df, title, min_n=0):
    df = df.copy()
    df['LIFECYCLE_STAGE'] = df['LIFECYCLE_STAGE'].str.strip()
    df['CANCELLATION_RATE_PCT'] = df['CANCELLATION_RATE_PCT'].astype(float)
    df['THRESHOLD'] = df['THRESHOLD'].astype(int)
    df['TOTAL'] = df['TOTAL'].astype(int)
    df = df.sort_values(['LIFECYCLE_SORT', 'THRESHOLD'])

    fig, ax = plt.subplots(figsize=(12, 7))
    texts = []

    for stage in lifecycle_order:
        stage_df = df[df['LIFECYCLE_STAGE'] == stage].copy()
        if min_n > 0:
            stage_df = stage_df[stage_df['TOTAL'] >= min_n]
        if stage_df.empty:
            continue
        ax.plot(
            stage_df['THRESHOLD'], stage_df['CANCELLATION_RATE_PCT'],
            marker='o', linewidth=2.2, markersize=7,
            label=stage, color=colors.get(stage, '#333'),
            linestyle=linestyles.get(stage, '-'),
        )
        for _, row in stage_df.iterrows():
            texts.append(ax.text(
                row['THRESHOLD'], row['CANCELLATION_RATE_PCT'],
                f"n={int(row['TOTAL']):,}",
                fontsize=7.5, color='#555',
            ))

    ax.axhline(y=14.3, color='#888', linestyle=':', linewidth=1.5, label='Average (14.3%)')
    ax.set_xticks(range(1, 11))
    ax.set_xticklabels([f'{i}+' for i in range(1, 11)])
    ax.set_xlabel("Minimum # of Meetings at Stage", fontsize=12, labelpad=10)
    ax.set_ylabel("Cancellation Rate (%)", fontsize=12, labelpad=10)
    ax.set_title(title, fontsize=13, pad=15, fontweight='bold')
    ax.legend(title="Lifecycle Stage", loc='upper right', fontsize=9, title_fontsize=10, handlelength=3.5)
    ax.grid(axis='y', alpha=0.3)
    sns.despine()
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='#aaa', lw=0.5))
    plt.tight_layout()
    plt.show()

In [ ]:
df_b2b = query(BASE_QUERY.format(filter="'b2b'"))
print(df_b2b.to_string(index=False))
plot_cancellation_rate(df_b2b, "Cancellation Rate for B2B Businesses with ≥ N Meetings at Each Lifecycle Stage")

In [ ]:
df_b2c = query(BASE_QUERY.format(filter="'b2c'"))
print(df_b2c.to_string(index=False))
plot_cancellation_rate(df_b2c, "Cancellation Rate for B2C Businesses with ≥ N Meetings at Each Lifecycle Stage")

In [ ]:
df_both = query(BASE_QUERY.format(filter="'both'"))
print(df_both.to_string(index=False))
plot_cancellation_rate(df_both, "Cancellation Rate for B2B+B2C Businesses with ≥ N Meetings at Each Lifecycle Stage")

In [ ]:
BASE_QUERY_SPLIT_BY_CUSTOMER_PARTNER = """
  WITH conversation_users AS (                                                                              
      SELECT                                                       
          a."conversation_id",                                                                              
          a."meeting_lifecycle",                                                                            
          SPLIT_PART(TRANSLATE(c."user_ids", '{{}}"', ''), ',', 1) AS user1,                                
          SPLIT_PART(TRANSLATE(c."user_ids", '{{}}"', ''), ',', 2) AS user2
      FROM OPENFLOW_DB."public"."conversation_meeting_analyses" a                                           
      INNER JOIN OPENFLOW_DB."public"."conversations" c           
          ON c."id" = a."conversation_id"                                                                   
  ),                                                              

  -- Resolve both users to business IDs in one pass                                                         
  conversation_businesses AS (
      SELECT                                                                                                
          cu."conversation_id",                                   
          cu."meeting_lifecycle",
          u1.BUSINESS_ID AS business1,
          u2.BUSINESS_ID AS business2
      FROM conversation_users cu                                                                            
      INNER JOIN ALIGNABLE_REPORTING_V2.CORE.DIM_USER u1
          ON u1.ID::VARCHAR = cu.user1 AND u1.IS_PRIMARY_USER = TRUE                                        
      INNER JOIN ALIGNABLE_REPORTING_V2.CORE.DIM_USER u2          
          ON u2.ID::VARCHAR = cu.user2 AND u2.IS_PRIMARY_USER = TRUE                                        
  ),
                                                                                                            
  -- Current taxonomy version                                                                               
  current_mir_version AS (
      SELECT MAX("version") AS v                                                                            
      FROM OPENFLOW_DB."public"."classification_mapped_industry_roles"
  ),

  -- Each business's desired customer/partner industry roles
  -- (custom tags → mapped industry roles)
  business_desired_roles AS (                                                                               
      SELECT DISTINCT
          bct."business_id",                                                                                
          bct."display_type",  -- 2 = customer, 3 = partner       
          mir."industry_role_id"                                                                            
      FROM OPENFLOW_DB."public"."classification_business_custom_tags" bct
      CROSS JOIN current_mir_version cmv                                                                    
      INNER JOIN OPENFLOW_DB."public"."classification_mapped_industry_roles" mir
          ON mir."custom_tag_id" = bct."custom_tag_id"                                                      
          AND mir."version" = cmv.v
      WHERE bct."active" = true                                                                             
        AND bct."display_type" IN (2, 3)                          
  ),                                                                                                        
   
  -- Each business's own industry roles                                                                     
  business_actual_roles AS (                                      
      SELECT "business_id", "industry_role_id"
      FROM OPENFLOW_DB."public"."classification_business_industry_roles"
  ),

  -- From business1's perspective: is business2 their ideal customer or partner?                            
  b1_ideal_flags AS (
      SELECT                                                                                                
          cb."conversation_id",                                   
          MAX(CASE WHEN bdr."display_type" = 2 THEN 1 ELSE 0 END) AS b2_is_customer_of_b1,                  
          MAX(CASE WHEN bdr."display_type" = 3 THEN 1 ELSE 0 END) AS b2_is_partner_of_b1
      FROM conversation_businesses cb                                                                       
      INNER JOIN business_actual_roles bar ON bar."business_id" = cb.business2
      INNER JOIN business_desired_roles bdr                                                                 
          ON bdr."business_id" = cb.business1                                                               
          AND bdr."industry_role_id" = bar."industry_role_id"
      GROUP BY 1                                                                                            
  ),                                                              

  -- From business2's perspective: is business1 their ideal customer or partner?                            
  b2_ideal_flags AS (
      SELECT                                                                                                
          cb."conversation_id",                                   
          MAX(CASE WHEN bdr."display_type" = 2 THEN 1 ELSE 0 END) AS b1_is_customer_of_b2,
          MAX(CASE WHEN bdr."display_type" = 3 THEN 1 ELSE 0 END) AS b1_is_partner_of_b2                    
      FROM conversation_businesses cb
      INNER JOIN business_actual_roles bar ON bar."business_id" = cb.business1                              
      INNER JOIN business_desired_roles bdr                                                                 
          ON bdr."business_id" = cb.business2
          AND bdr."industry_role_id" = bar."industry_role_id"                                               
      GROUP BY 1                                                                                            
  ),
                                                                                                            
  -- Combine flags back onto conversations                        
  conversation_ideal_flags AS (
      SELECT                                                                                                
          cb."conversation_id",
          cb."meeting_lifecycle",                                                                           
          cb.business1,                                           
          cb.business2,
          COALESCE(b1f.b2_is_customer_of_b1, 0) AS b2_is_customer_of_b1,
          COALESCE(b1f.b2_is_partner_of_b1, 0) AS b2_is_partner_of_b1,                                      
          COALESCE(b2f.b1_is_customer_of_b2, 0) AS b1_is_customer_of_b2,
          COALESCE(b2f.b1_is_partner_of_b2, 0) AS b1_is_partner_of_b2                                       
      FROM conversation_businesses cb                             
      LEFT JOIN b1_ideal_flags b1f ON b1f."conversation_id" = cb."conversation_id"                          
      LEFT JOIN b2_ideal_flags b2f ON b2f."conversation_id" = cb."conversation_id"
  ),                                                                                                        
                                                                                                            
  -- Unpivot: each user gets a row per matching relationship type
  meeting_relationships AS (                                                                                
      SELECT business1 AS business_id, "meeting_lifecycle", 'with_customer' AS relationship_type
      FROM conversation_ideal_flags WHERE b2_is_customer_of_b1 = 1
      UNION ALL                                                                                             
      SELECT business1, "meeting_lifecycle", 'with_partner'
      FROM conversation_ideal_flags WHERE b2_is_partner_of_b1 = 1                                           
      UNION ALL                                                                                             
      SELECT business2, "meeting_lifecycle", 'with_customer'
      FROM conversation_ideal_flags WHERE b1_is_customer_of_b2 = 1                                          
      UNION ALL                                                   
      SELECT business2, "meeting_lifecycle", 'with_partner'
      FROM conversation_ideal_flags WHERE b1_is_partner_of_b2 = 1                                           
  ),
                                                                                                            
  business_stage_counts AS (                                      
      SELECT
          business_id AS BUSINESS_ID,
          "meeting_lifecycle" AS MEETING_LIFECYCLE,
          relationship_type,                                                                                
          COUNT(*) AS meetings_at_stage
      FROM meeting_relationships                                                                            
      GROUP BY 1, 2, 3                                            
  ),

  business_customer_type AS (
      SELECT
          b.BUSINESS_ID,
          CASE                                                                                              
              WHEN rcm."customer_type" = 1 THEN 'b2b'
              WHEN rcm."customer_type" = 0 THEN 'b2c'                                                       
              WHEN rcm."customer_type" = 2 THEN 'both'            
              WHEN rcm."customer_type" = 3 THEN 'none'                                                      
              WHEN bf."customer_type" = 0 THEN 'b2b'
              WHEN bf."customer_type" = 1 THEN 'b2c'                                                        
              ELSE 'unknown'                                                                                
          END AS business_type
      FROM (SELECT DISTINCT BUSINESS_ID FROM business_stage_counts) b                                       
      LEFT JOIN OPENFLOW_DB."public"."relationships_business_customer_metadata" rcm
          ON rcm."business_id" = b.BUSINESS_ID                                                              
      LEFT JOIN OPENFLOW_DB."public"."business_flags" bf
          ON bf."business_id" = b.BUSINESS_ID
      QUALIFY ROW_NUMBER() OVER (PARTITION BY b.BUSINESS_ID ORDER BY rcm."customer_type" NULLS LAST) = 1
  ),                                                              

  thresholds AS (
      SELECT column1 AS threshold FROM VALUES (1),(2),(3),(4),(5),(6),(7),(8),(9),(10)
  ),                                                                                                        
   
  cumulative AS (                                                                                           
      SELECT                                                      
          b.MEETING_LIFECYCLE,
          b.relationship_type,
          t.threshold,
          b.BUSINESS_ID
      FROM business_stage_counts b                                                                          
      CROSS JOIN thresholds t
      INNER JOIN business_customer_type bct ON bct.BUSINESS_ID = b.BUSINESS_ID                              
      WHERE b.meetings_at_stage >= t.threshold                    
        AND bct.business_type IN ({filter})                                                                 
  )
                                                                                                            
  SELECT                                                          
      CASE c.MEETING_LIFECYCLE
          WHEN 6 THEN 'completed'
          WHEN 5 THEN 'scheduled'
          WHEN 4 THEN 'mutual_intent'
      END || ' ' || c.relationship_type || ' (' || c.MEETING_LIFECYCLE || ')' AS lifecycle_stage,           
      c.MEETING_LIFECYCLE AS lifecycle_sort,
      c.relationship_type,                                                                                  
      c.threshold,                                                
      COUNT_IF(s.SUBSCRIPTION_STATUS = 'churned') AS churned,                                               
      COUNT_IF(s.SUBSCRIPTION_STATUS = 'active') AS active,
      churned + active AS total,                                                                            
      ROUND(churned / NULLIF(churned + active, 0) * 100, 2) AS cancellation_rate_pct
  FROM ALIGNABLE_REPORTING_V2.MEMBERSHIPS.DIM_MEMBERSHIP_SUBSCRIPTIONS s                                    
  INNER JOIN cumulative c ON c.BUSINESS_ID = s.BUSINESS_ID
  WHERE c.MEETING_LIFECYCLE IN (4, 5, 6)                                                                    
    AND s.ACTIVE_AT >= GREATEST('2026-01-20'::DATE, DATEADD(DAY, -90, CURRENT_DATE()))
  GROUP BY 1, 2, 3, 4                                                                                       
  ORDER BY lifecycle_sort DESC, relationship_type, threshold ASC
"""

In [ ]:
_stage_rename = {
    'completed with_customer (6)': 'completed with customer',
    'completed with_partner (6)': 'completed with partner',
    'scheduled with_customer (5)': 'scheduled with customer',
    'scheduled with_partner (5)': 'scheduled with partner',
    'mutual_intent with_customer (4)': 'mutual intent with customer',
    'mutual_intent with_partner (4)': 'mutual intent with partner',
}

lifecycle_order_split = [
    'completed with customer',
    'completed with partner',
    'scheduled with customer',
    'scheduled with partner',
    'mutual intent with customer',
    'mutual intent with partner',
]

colors_split = {
    'completed with customer': '#27ae60',
    'completed with partner':  '#27ae60',
    'scheduled with customer': '#2471a3',
    'scheduled with partner':  '#2471a3',
    'mutual intent with customer': '#7d3c98',
    'mutual intent with partner':  '#7d3c98',
}

linestyles_split = {
    'completed with customer': '-',
    'completed with partner':  '--',
    'scheduled with customer': '-',
    'scheduled with partner':  '--',
    'mutual intent with customer': '-',
    'mutual intent with partner':  '--',
}

# Temporarily swap globals so plot_cancellation_rate uses the split labels
_orig_lifecycle_order = lifecycle_order
_orig_colors = colors
_orig_linestyles = globals().get('linestyles', {})
lifecycle_order = lifecycle_order_split
colors = colors_split
linestyles = linestyles_split

In [ ]:
df_b2b_split = query(BASE_QUERY_SPLIT_BY_CUSTOMER_PARTNER.format(filter="'b2b'"))
df_b2b_split['LIFECYCLE_STAGE'] = df_b2b_split['LIFECYCLE_STAGE'].str.strip().map(_stage_rename)
print(df_b2b_split.to_string(index=False))
plot_cancellation_rate(df_b2b_split, "Cancellation Rate for B2B Businesses by Customer/Partner × Lifecycle Stage (≥ N Meetings)", min_n=5)

In [ ]:
df_b2c_split = query(BASE_QUERY_SPLIT_BY_CUSTOMER_PARTNER.format(filter="'b2c'"))
df_b2c_split['LIFECYCLE_STAGE'] = df_b2c_split['LIFECYCLE_STAGE'].str.strip().map(_stage_rename)
print(df_b2c_split.to_string(index=False))
plot_cancellation_rate(df_b2c_split, "Cancellation Rate for B2C Businesses by Customer/Partner × Lifecycle Stage (≥ N Meetings)", min_n=5)

In [ ]:
QUERY_WITH_UNKNOWN = """
  WITH conversation_users AS (
      SELECT
          a."conversation_id",
          a."meeting_lifecycle",
          SPLIT_PART(TRANSLATE(c."user_ids", '{{}}"', ''), ',', 1) AS user1,
          SPLIT_PART(TRANSLATE(c."user_ids", '{{}}"', ''), ',', 2) AS user2
      FROM OPENFLOW_DB."public"."conversation_meeting_analyses" a
      INNER JOIN OPENFLOW_DB."public"."conversations" c
          ON c."id" = a."conversation_id"
  ),

  conversation_businesses AS (
      SELECT
          cu."conversation_id",
          cu."meeting_lifecycle",
          u1.BUSINESS_ID AS business1,
          u2.BUSINESS_ID AS business2
      FROM conversation_users cu
      INNER JOIN ALIGNABLE_REPORTING_V2.CORE.DIM_USER u1
          ON u1.ID::VARCHAR = cu.user1 AND u1.IS_PRIMARY_USER = TRUE
      INNER JOIN ALIGNABLE_REPORTING_V2.CORE.DIM_USER u2
          ON u2.ID::VARCHAR = cu.user2 AND u2.IS_PRIMARY_USER = TRUE
  ),

  current_mir_version AS (
      SELECT MAX("version") AS v
      FROM OPENFLOW_DB."public"."classification_mapped_industry_roles"
  ),

  business_desired_roles AS (
      SELECT DISTINCT
          bct."business_id",
          bct."display_type",
          mir."industry_role_id"
      FROM OPENFLOW_DB."public"."classification_business_custom_tags" bct
      CROSS JOIN current_mir_version cmv
      INNER JOIN OPENFLOW_DB."public"."classification_mapped_industry_roles" mir
          ON mir."custom_tag_id" = bct."custom_tag_id"
          AND mir."version" = cmv.v
      WHERE bct."active" = true
        AND bct."display_type" IN (2, 3)
  ),

  business_actual_roles AS (
      SELECT "business_id", "industry_role_id"
      FROM OPENFLOW_DB."public"."classification_business_industry_roles"
  ),

  b1_ideal_flags AS (
      SELECT
          cb."conversation_id",
          MAX(CASE WHEN bdr."display_type" = 2 THEN 1 ELSE 0 END) AS b2_is_customer_of_b1,
          MAX(CASE WHEN bdr."display_type" = 3 THEN 1 ELSE 0 END) AS b2_is_partner_of_b1
      FROM conversation_businesses cb
      INNER JOIN business_actual_roles bar ON bar."business_id" = cb.business2
      INNER JOIN business_desired_roles bdr
          ON bdr."business_id" = cb.business1
          AND bdr."industry_role_id" = bar."industry_role_id"
      GROUP BY 1
  ),

  b2_ideal_flags AS (
      SELECT
          cb."conversation_id",
          MAX(CASE WHEN bdr."display_type" = 2 THEN 1 ELSE 0 END) AS b1_is_customer_of_b2,
          MAX(CASE WHEN bdr."display_type" = 3 THEN 1 ELSE 0 END) AS b1_is_partner_of_b2
      FROM conversation_businesses cb
      INNER JOIN business_actual_roles bar ON bar."business_id" = cb.business1
      INNER JOIN business_desired_roles bdr
          ON bdr."business_id" = cb.business2
          AND bdr."industry_role_id" = bar."industry_role_id"
      GROUP BY 1
  ),

  conversation_ideal_flags AS (
      SELECT
          cb."conversation_id",
          cb."meeting_lifecycle",
          cb.business1,
          cb.business2,
          COALESCE(b1f.b2_is_customer_of_b1, 0) AS b2_is_customer_of_b1,
          COALESCE(b1f.b2_is_partner_of_b1, 0) AS b2_is_partner_of_b1,
          COALESCE(b2f.b1_is_customer_of_b2, 0) AS b1_is_customer_of_b2,
          COALESCE(b2f.b1_is_partner_of_b2, 0) AS b1_is_partner_of_b2
      FROM conversation_businesses cb
      LEFT JOIN b1_ideal_flags b1f ON b1f."conversation_id" = cb."conversation_id"
      LEFT JOIN b2_ideal_flags b2f ON b2f."conversation_id" = cb."conversation_id"
  ),

  meeting_relationships AS (
      SELECT business1 AS business_id, "meeting_lifecycle", 'with_customer' AS relationship_type
      FROM conversation_ideal_flags WHERE b2_is_customer_of_b1 = 1
      UNION ALL
      SELECT business1, "meeting_lifecycle", 'with_partner'
      FROM conversation_ideal_flags WHERE b2_is_partner_of_b1 = 1
      UNION ALL
      SELECT business2, "meeting_lifecycle", 'with_customer'
      FROM conversation_ideal_flags WHERE b1_is_customer_of_b2 = 1
      UNION ALL
      SELECT business2, "meeting_lifecycle", 'with_partner'
      FROM conversation_ideal_flags WHERE b1_is_partner_of_b2 = 1
      UNION ALL
      SELECT business1, "meeting_lifecycle", 'with_unknown'
      FROM conversation_ideal_flags WHERE b2_is_customer_of_b1 = 0 AND b2_is_partner_of_b1 = 0
      UNION ALL
      SELECT business2, "meeting_lifecycle", 'with_unknown'
      FROM conversation_ideal_flags WHERE b1_is_customer_of_b2 = 0 AND b1_is_partner_of_b2 = 0
  ),

  business_stage_counts AS (
      SELECT
          business_id AS BUSINESS_ID,
          "meeting_lifecycle" AS MEETING_LIFECYCLE,
          relationship_type,
          COUNT(*) AS meetings_at_stage
      FROM meeting_relationships
      GROUP BY 1, 2, 3
  ),

  business_customer_type AS (
      SELECT
          b.BUSINESS_ID,
          CASE
              WHEN rcm."customer_type" = 1 THEN 'b2b'
              WHEN rcm."customer_type" = 0 THEN 'b2c'
              WHEN rcm."customer_type" = 2 THEN 'both'
              WHEN rcm."customer_type" = 3 THEN 'none'
              WHEN bf."customer_type" = 0 THEN 'b2b'
              WHEN bf."customer_type" = 1 THEN 'b2c'
              ELSE 'unknown'
          END AS business_type
      FROM (SELECT DISTINCT BUSINESS_ID FROM business_stage_counts) b
      LEFT JOIN OPENFLOW_DB."public"."relationships_business_customer_metadata" rcm
          ON rcm."business_id" = b.BUSINESS_ID
      LEFT JOIN OPENFLOW_DB."public"."business_flags" bf
          ON bf."business_id" = b.BUSINESS_ID
      QUALIFY ROW_NUMBER() OVER (PARTITION BY b.BUSINESS_ID ORDER BY rcm."customer_type" NULLS LAST) = 1
  ),

  thresholds AS (
      SELECT column1 AS threshold FROM VALUES (1),(2),(3),(4),(5),(6),(7),(8),(9),(10)
  ),

  cumulative AS (
      SELECT
          b.MEETING_LIFECYCLE,
          b.relationship_type,
          t.threshold,
          b.BUSINESS_ID
      FROM business_stage_counts b
      CROSS JOIN thresholds t
      INNER JOIN business_customer_type bct ON bct.BUSINESS_ID = b.BUSINESS_ID
      WHERE b.meetings_at_stage >= t.threshold
        AND bct.business_type IN ({filter})
  )

  SELECT
      CASE c.MEETING_LIFECYCLE
          WHEN 6 THEN 'completed'
          WHEN 5 THEN 'scheduled'
          WHEN 4 THEN 'mutual_intent'
      END || ' ' || c.relationship_type AS lifecycle_stage,
      c.MEETING_LIFECYCLE AS lifecycle_sort,
      c.relationship_type,
      c.threshold,
      COUNT_IF(s.SUBSCRIPTION_STATUS = 'churned') AS churned,
      COUNT_IF(s.SUBSCRIPTION_STATUS = 'active') AS active,
      churned + active AS total,
      ROUND(churned / NULLIF(churned + active, 0) * 100, 2) AS cancellation_rate_pct
  FROM ALIGNABLE_REPORTING_V2.MEMBERSHIPS.DIM_MEMBERSHIP_SUBSCRIPTIONS s
  INNER JOIN cumulative c ON c.BUSINESS_ID = s.BUSINESS_ID
  WHERE c.MEETING_LIFECYCLE IN (4, 5, 6)
    AND s.ACTIVE_AT >= GREATEST('2026-01-20'::DATE, DATEADD(DAY, -90, CURRENT_DATE()))
  GROUP BY 1, 2, 3, 4
  ORDER BY lifecycle_sort DESC, relationship_type, threshold ASC
"""

_unknown_lo = [
    'completed with customer', 'completed with partner', 'completed with unknown',
    'scheduled with customer', 'scheduled with partner', 'scheduled with unknown',
]
_unknown_co = {
    'completed with customer': '#27ae60', 'completed with partner': '#27ae60', 'completed with unknown': '#27ae60',
    'scheduled with customer': '#2471a3', 'scheduled with partner': '#2471a3', 'scheduled with unknown': '#2471a3',
}
_unknown_ls = {
    'completed with customer': '-', 'completed with partner': '--', 'completed with unknown': ':',
    'scheduled with customer': '-', 'scheduled with partner': '--', 'scheduled with unknown': ':',
}

def plot_with_unknown(df, title):
    df = df.copy()
    df['LIFECYCLE_STAGE'] = df['LIFECYCLE_STAGE'].str.strip().str.replace('_', ' ')
    _save_lo, _save_co, _save_ls = lifecycle_order, colors, globals().get('linestyles', {})
    globals()['lifecycle_order'] = _unknown_lo
    globals()['colors'] = _unknown_co
    globals()['linestyles'] = _unknown_ls
    plot_cancellation_rate(df, title, min_n=5)
    globals()['lifecycle_order'] = _save_lo
    globals()['colors'] = _save_co
    globals()['linestyles'] = _save_ls

df_b2c_with_unknown = query(QUERY_WITH_UNKNOWN.format(filter="'b2c'"))
print(df_b2c_with_unknown.to_string(index=False))
plot_with_unknown(df_b2c_with_unknown, "Cancellation Rate for B2C Businesses by Customer/Partner/Unknown × Lifecycle Stage (≥ N Meetings)")

In [ ]:
UNKNOWN_DETAIL_QUERY = """
WITH conversation_users AS (
    SELECT
        a."conversation_id",
        a."meeting_lifecycle",
        SPLIT_PART(TRANSLATE(c."user_ids", '{{}}"', ''), ',', 1) AS user1,
        SPLIT_PART(TRANSLATE(c."user_ids", '{{}}"', ''), ',', 2) AS user2
    FROM OPENFLOW_DB."public"."conversation_meeting_analyses" a
    INNER JOIN OPENFLOW_DB."public"."conversations" c
        ON c."id" = a."conversation_id"
),

conversation_businesses AS (
    SELECT
        cu."conversation_id",
        cu."meeting_lifecycle",
        u1.BUSINESS_ID AS business1,
        u2.BUSINESS_ID AS business2
    FROM conversation_users cu
    INNER JOIN ALIGNABLE_REPORTING_V2.CORE.DIM_USER u1
        ON u1.ID::VARCHAR = cu.user1 AND u1.IS_PRIMARY_USER = TRUE
    INNER JOIN ALIGNABLE_REPORTING_V2.CORE.DIM_USER u2
        ON u2.ID::VARCHAR = cu.user2 AND u2.IS_PRIMARY_USER = TRUE
),

current_mir_version AS (
    SELECT MAX("version") AS v
    FROM OPENFLOW_DB."public"."classification_mapped_industry_roles"
),

business_desired_roles AS (
    SELECT DISTINCT
        bct."business_id",
        bct."display_type",
        mir."industry_role_id"
    FROM OPENFLOW_DB."public"."classification_business_custom_tags" bct
    CROSS JOIN current_mir_version cmv
    INNER JOIN OPENFLOW_DB."public"."classification_mapped_industry_roles" mir
        ON mir."custom_tag_id" = bct."custom_tag_id"
        AND mir."version" = cmv.v
    WHERE bct."active" = true
      AND bct."display_type" IN (2, 3)
),

business_actual_roles AS (
    SELECT "business_id", "industry_role_id"
    FROM OPENFLOW_DB."public"."classification_business_industry_roles"
),

-- Flag whether each business has ANY desired roles configured
business_has_desired_roles AS (
    SELECT
        "business_id",
        MAX(CASE WHEN "display_type" = 2 THEN 1 ELSE 0 END) AS has_customer_prefs,
        MAX(CASE WHEN "display_type" = 3 THEN 1 ELSE 0 END) AS has_partner_prefs
    FROM (
        SELECT DISTINCT bct."business_id", bct."display_type"
        FROM OPENFLOW_DB."public"."classification_business_custom_tags" bct
        WHERE bct."active" = true AND bct."display_type" IN (2, 3)
    )
    GROUP BY 1
),

-- Desired customer roles (display_type=2) as comma-separated "name id" strings
business_desired_customer_roles_agg AS (
    SELECT
        bdr."business_id",
        LISTAGG(DISTINCT ir."name" || ' ' || ir."id", ', ') AS desired_customer_roles
    FROM business_desired_roles bdr
    INNER JOIN OPENFLOW_DB."public"."classification_industry_roles" ir
        ON ir."id" = bdr."industry_role_id"
    WHERE bdr."display_type" = 2
    GROUP BY 1
),

-- Desired partner roles (display_type=3) as comma-separated "name id" strings
business_desired_partner_roles_agg AS (
    SELECT
        bdr."business_id",
        LISTAGG(DISTINCT ir."name" || ' ' || ir."id", ', ') AS desired_partner_roles
    FROM business_desired_roles bdr
    INNER JOIN OPENFLOW_DB."public"."classification_industry_roles" ir
        ON ir."id" = bdr."industry_role_id"
    WHERE bdr."display_type" = 3
    GROUP BY 1
),

-- Actual roles as comma-separated "name id" strings
business_actual_roles_agg AS (
    SELECT
        bar."business_id",
        LISTAGG(DISTINCT ir."name" || ' ' || ir."id", ', ') AS actual_roles
    FROM business_actual_roles bar
    INNER JOIN OPENFLOW_DB."public"."classification_industry_roles" ir
        ON ir."id" = bar."industry_role_id"
    GROUP BY 1
),

b1_ideal_flags AS (
    SELECT
        cb."conversation_id",
        MAX(CASE WHEN bdr."display_type" = 2 THEN 1 ELSE 0 END) AS b2_is_customer_of_b1,
        MAX(CASE WHEN bdr."display_type" = 3 THEN 1 ELSE 0 END) AS b2_is_partner_of_b1
    FROM conversation_businesses cb
    INNER JOIN business_actual_roles bar ON bar."business_id" = cb.business2
    INNER JOIN business_desired_roles bdr
        ON bdr."business_id" = cb.business1
        AND bdr."industry_role_id" = bar."industry_role_id"
    GROUP BY 1
),

b2_ideal_flags AS (
    SELECT
        cb."conversation_id",
        MAX(CASE WHEN bdr."display_type" = 2 THEN 1 ELSE 0 END) AS b1_is_customer_of_b2,
        MAX(CASE WHEN bdr."display_type" = 3 THEN 1 ELSE 0 END) AS b1_is_partner_of_b2
    FROM conversation_businesses cb
    INNER JOIN business_actual_roles bar ON bar."business_id" = cb.business1
    INNER JOIN business_desired_roles bdr
        ON bdr."business_id" = cb.business2
        AND bdr."industry_role_id" = bar."industry_role_id"
    GROUP BY 1
),

conversation_ideal_flags AS (
    SELECT
        cb."conversation_id",
        cb."meeting_lifecycle",
        cb.business1,
        cb.business2,
        COALESCE(b1f.b2_is_customer_of_b1, 0) AS b2_is_customer_of_b1,
        COALESCE(b1f.b2_is_partner_of_b1, 0) AS b2_is_partner_of_b1,
        COALESCE(b2f.b1_is_customer_of_b2, 0) AS b1_is_customer_of_b2,
        COALESCE(b2f.b1_is_partner_of_b2, 0) AS b1_is_partner_of_b2
    FROM conversation_businesses cb
    LEFT JOIN b1_ideal_flags b1f ON b1f."conversation_id" = cb."conversation_id"
    LEFT JOIN b2_ideal_flags b2f ON b2f."conversation_id" = cb."conversation_id"
),

-- From business1's perspective: business2 is unknown to business1
unknown_relationships AS (
    SELECT
        business1 AS source_business_id,
        business2 AS recipient_business_id,
        "meeting_lifecycle",
        'with_unknown' AS relationship_type
    FROM conversation_ideal_flags
    WHERE b2_is_customer_of_b1 = 0 AND b2_is_partner_of_b1 = 0
    UNION ALL
    -- From business2's perspective: business1 is unknown to business2
    SELECT
        business2 AS source_business_id,
        business1 AS recipient_business_id,
        "meeting_lifecycle",
        'with_unknown' AS relationship_type
    FROM conversation_ideal_flags
    WHERE b1_is_customer_of_b2 = 0 AND b1_is_partner_of_b2 = 0
),

business_customer_type AS (
    SELECT
        b.source_business_id AS BUSINESS_ID,
        CASE
            WHEN rcm."customer_type" = 1 THEN 'b2b'
            WHEN rcm."customer_type" = 0 THEN 'b2c'
            WHEN rcm."customer_type" = 2 THEN 'both'
            WHEN rcm."customer_type" = 3 THEN 'none'
            WHEN bf."customer_type" = 0 THEN 'b2b'
            WHEN bf."customer_type" = 1 THEN 'b2c'
            ELSE 'unknown'
        END AS business_type
    FROM (SELECT DISTINCT source_business_id FROM unknown_relationships) b
    LEFT JOIN OPENFLOW_DB."public"."relationships_business_customer_metadata" rcm
        ON rcm."business_id" = b.source_business_id
    LEFT JOIN OPENFLOW_DB."public"."business_flags" bf
        ON bf."business_id" = b.source_business_id
    QUALIFY ROW_NUMBER() OVER (PARTITION BY b.source_business_id ORDER BY rcm."customer_type" NULLS LAST) = 1
),

-- Aggregate meetings per source-recipient pair at each lifecycle stage
pair_stage_counts AS (
    SELECT
        ur.source_business_id,
        ur.recipient_business_id,
        ur.relationship_type,
        COUNT_IF(ur."meeting_lifecycle" = 6) AS completed_meetings,
        COUNT_IF(ur."meeting_lifecycle" = 5) AS scheduled_meetings,
        COUNT_IF(ur."meeting_lifecycle" = 4) AS mutual_intent_meetings
    FROM unknown_relationships ur
    INNER JOIN business_customer_type bct ON bct.BUSINESS_ID = ur.source_business_id
    WHERE bct.business_type IN ({filter})
    GROUP BY 1, 2, 3
)

SELECT
    psc.source_business_id,
    b_src."name" AS source_business_name,
    psc.recipient_business_id,
    b_rcpt."name" AS recipient_business_name,
    psc.relationship_type,
    psc.completed_meetings,
    psc.scheduled_meetings,
    psc.mutual_intent_meetings,
    s.SUBSCRIPTION_STATUS AS source_subscription_status,
    COALESCE(hdr_src.has_customer_prefs, 0) AS source_has_customer_prefs,
    COALESCE(hdr_src.has_partner_prefs, 0) AS source_has_partner_prefs,
    COALESCE(hdr_rcpt.has_customer_prefs, 0) AS recipient_has_customer_prefs,
    COALESCE(hdr_rcpt.has_partner_prefs, 0) AS recipient_has_partner_prefs,
    dcr_src.desired_customer_roles AS source_desired_customer_roles,
    dpr_src.desired_partner_roles AS source_desired_partner_roles,
    ar_src.actual_roles AS source_actual_roles,
    dcr_rcpt.desired_customer_roles AS recipient_desired_customer_roles,
    dpr_rcpt.desired_partner_roles AS recipient_desired_partner_roles,
    ar_rcpt.actual_roles AS recipient_actual_roles
FROM pair_stage_counts psc
LEFT JOIN OPENFLOW_DB."public"."businesses" b_src
    ON b_src."id" = psc.source_business_id
LEFT JOIN OPENFLOW_DB."public"."businesses" b_rcpt
    ON b_rcpt."id" = psc.recipient_business_id
LEFT JOIN ALIGNABLE_REPORTING_V2.MEMBERSHIPS.DIM_MEMBERSHIP_SUBSCRIPTIONS s
    ON s.BUSINESS_ID = psc.source_business_id
    AND s.ACTIVE_AT >= GREATEST('2026-01-20'::DATE, DATEADD(DAY, -90, CURRENT_DATE()))
LEFT JOIN business_has_desired_roles hdr_src
    ON hdr_src."business_id" = psc.source_business_id
LEFT JOIN business_has_desired_roles hdr_rcpt
    ON hdr_rcpt."business_id" = psc.recipient_business_id
LEFT JOIN business_desired_customer_roles_agg dcr_src
    ON dcr_src."business_id" = psc.source_business_id
LEFT JOIN business_desired_partner_roles_agg dpr_src
    ON dpr_src."business_id" = psc.source_business_id
LEFT JOIN business_actual_roles_agg ar_src
    ON ar_src."business_id" = psc.source_business_id
LEFT JOIN business_desired_customer_roles_agg dcr_rcpt
    ON dcr_rcpt."business_id" = psc.recipient_business_id
LEFT JOIN business_desired_partner_roles_agg dpr_rcpt
    ON dpr_rcpt."business_id" = psc.recipient_business_id
LEFT JOIN business_actual_roles_agg ar_rcpt
    ON ar_rcpt."business_id" = psc.recipient_business_id
ORDER BY psc.completed_meetings DESC, psc.scheduled_meetings DESC
"""

df_unknown_b2c_detail = query(UNKNOWN_DETAIL_QUERY.format(filter="'b2c'"))
print(f"Rows: {len(df_unknown_b2c_detail)}")
df_unknown_b2c_detail.head(1)


In [ ]:
UNKNOWN_ROLE_BREAKDOWN_QUERY = """
WITH conversation_users AS (
    SELECT
        a."conversation_id",
        a."meeting_lifecycle",
        SPLIT_PART(TRANSLATE(c."user_ids", '{{}}"', ''), ',', 1) AS user1,
        SPLIT_PART(TRANSLATE(c."user_ids", '{{}}"', ''), ',', 2) AS user2
    FROM OPENFLOW_DB."public"."conversation_meeting_analyses" a
    INNER JOIN OPENFLOW_DB."public"."conversations" c
        ON c."id" = a."conversation_id"
),

conversation_businesses AS (
    SELECT
        cu."conversation_id",
        cu."meeting_lifecycle",
        u1.BUSINESS_ID AS business1,
        u2.BUSINESS_ID AS business2
    FROM conversation_users cu
    INNER JOIN ALIGNABLE_REPORTING_V2.CORE.DIM_USER u1
        ON u1.ID::VARCHAR = cu.user1 AND u1.IS_PRIMARY_USER = TRUE
    INNER JOIN ALIGNABLE_REPORTING_V2.CORE.DIM_USER u2
        ON u2.ID::VARCHAR = cu.user2 AND u2.IS_PRIMARY_USER = TRUE
),

current_mir_version AS (
    SELECT MAX("version") AS v
    FROM OPENFLOW_DB."public"."classification_mapped_industry_roles"
),

business_desired_roles AS (
    SELECT DISTINCT
        bct."business_id",
        bct."display_type",
        mir."industry_role_id"
    FROM OPENFLOW_DB."public"."classification_business_custom_tags" bct
    CROSS JOIN current_mir_version cmv
    INNER JOIN OPENFLOW_DB."public"."classification_mapped_industry_roles" mir
        ON mir."custom_tag_id" = bct."custom_tag_id"
        AND mir."version" = cmv.v
    WHERE bct."active" = true
      AND bct."display_type" IN (2, 3)
),

business_actual_roles AS (
    SELECT "business_id", "industry_role_id"
    FROM OPENFLOW_DB."public"."classification_business_industry_roles"
),

business_desired_customer_roles_agg AS (
    SELECT
        bdr."business_id",
        LISTAGG(DISTINCT ir."name" || ' ' || ir."id", ', ') AS desired_customer_roles,
        COUNT(DISTINCT bdr."industry_role_id") AS desired_customer_role_count
    FROM business_desired_roles bdr
    INNER JOIN OPENFLOW_DB."public"."classification_industry_roles" ir
        ON ir."id" = bdr."industry_role_id"
    WHERE bdr."display_type" = 2
    GROUP BY 1
),

business_desired_partner_roles_agg AS (
    SELECT
        bdr."business_id",
        LISTAGG(DISTINCT ir."name" || ' ' || ir."id", ', ') AS desired_partner_roles,
        COUNT(DISTINCT bdr."industry_role_id") AS desired_partner_role_count
    FROM business_desired_roles bdr
    INNER JOIN OPENFLOW_DB."public"."classification_industry_roles" ir
        ON ir."id" = bdr."industry_role_id"
    WHERE bdr."display_type" = 3
    GROUP BY 1
),

b1_ideal_flags AS (
    SELECT
        cb."conversation_id",
        MAX(CASE WHEN bdr."display_type" = 2 THEN 1 ELSE 0 END) AS b2_is_customer_of_b1,
        MAX(CASE WHEN bdr."display_type" = 3 THEN 1 ELSE 0 END) AS b2_is_partner_of_b1
    FROM conversation_businesses cb
    INNER JOIN business_actual_roles bar ON bar."business_id" = cb.business2
    INNER JOIN business_desired_roles bdr
        ON bdr."business_id" = cb.business1
        AND bdr."industry_role_id" = bar."industry_role_id"
    GROUP BY 1
),

b2_ideal_flags AS (
    SELECT
        cb."conversation_id",
        MAX(CASE WHEN bdr."display_type" = 2 THEN 1 ELSE 0 END) AS b1_is_customer_of_b2,
        MAX(CASE WHEN bdr."display_type" = 3 THEN 1 ELSE 0 END) AS b1_is_partner_of_b2
    FROM conversation_businesses cb
    INNER JOIN business_actual_roles bar ON bar."business_id" = cb.business1
    INNER JOIN business_desired_roles bdr
        ON bdr."business_id" = cb.business2
        AND bdr."industry_role_id" = bar."industry_role_id"
    GROUP BY 1
),

conversation_ideal_flags AS (
    SELECT
        cb."conversation_id",
        cb."meeting_lifecycle",
        cb.business1,
        cb.business2,
        COALESCE(b1f.b2_is_customer_of_b1, 0) AS b2_is_customer_of_b1,
        COALESCE(b1f.b2_is_partner_of_b1, 0) AS b2_is_partner_of_b1,
        COALESCE(b2f.b1_is_customer_of_b2, 0) AS b1_is_customer_of_b2,
        COALESCE(b2f.b1_is_partner_of_b2, 0) AS b1_is_partner_of_b2
    FROM conversation_businesses cb
    LEFT JOIN b1_ideal_flags b1f ON b1f."conversation_id" = cb."conversation_id"
    LEFT JOIN b2_ideal_flags b2f ON b2f."conversation_id" = cb."conversation_id"
),

unknown_relationships AS (
    SELECT
        business1 AS source_business_id,
        business2 AS recipient_business_id,
        "meeting_lifecycle"
    FROM conversation_ideal_flags
    WHERE b2_is_customer_of_b1 = 0 AND b2_is_partner_of_b1 = 0
    UNION ALL
    SELECT
        business2 AS source_business_id,
        business1 AS recipient_business_id,
        "meeting_lifecycle"
    FROM conversation_ideal_flags
    WHERE b1_is_customer_of_b2 = 0 AND b1_is_partner_of_b2 = 0
),

business_customer_type AS (
    SELECT
        b.source_business_id AS BUSINESS_ID,
        CASE
            WHEN rcm."customer_type" = 1 THEN 'b2b'
            WHEN rcm."customer_type" = 0 THEN 'b2c'
            WHEN rcm."customer_type" = 2 THEN 'both'
            WHEN rcm."customer_type" = 3 THEN 'none'
            WHEN bf."customer_type" = 0 THEN 'b2b'
            WHEN bf."customer_type" = 1 THEN 'b2c'
            ELSE 'unknown'
        END AS business_type
    FROM (SELECT DISTINCT source_business_id FROM unknown_relationships) b
    LEFT JOIN OPENFLOW_DB."public"."relationships_business_customer_metadata" rcm
        ON rcm."business_id" = b.source_business_id
    LEFT JOIN OPENFLOW_DB."public"."business_flags" bf
        ON bf."business_id" = b.source_business_id
    QUALIFY ROW_NUMBER() OVER (PARTITION BY b.source_business_id ORDER BY rcm."customer_type" NULLS LAST) = 1
),

unknown_meeting_roles AS (
    SELECT
        ur.source_business_id,
        ur.recipient_business_id,
        ur."meeting_lifecycle",
        bar."industry_role_id" AS recipient_role_id
    FROM unknown_relationships ur
    INNER JOIN business_customer_type bct ON bct.BUSINESS_ID = ur.source_business_id
    INNER JOIN business_actual_roles bar ON bar."business_id" = ur.recipient_business_id
    WHERE bct.business_type IN ({filter})
      AND ur."meeting_lifecycle" IN (4, 5, 6)
)

SELECT
    umr.source_business_id,
    b_src."name" AS source_business_name,
    ir."name" || ' ' || ir."id" AS recipient_role,
    COUNT(*) AS meeting_count,
    COUNT_IF(umr."meeting_lifecycle" = 6) AS completed_meetings,
    COUNT_IF(umr."meeting_lifecycle" = 5) AS scheduled_meetings,
    COUNT_IF(umr."meeting_lifecycle" = 4) AS mutual_intent_meetings,
    MAX(CASE WHEN bdr_cust."industry_role_id" IS NOT NULL THEN 1 ELSE 0 END) AS is_desired_customer,
    MAX(CASE WHEN bdr_part."industry_role_id" IS NOT NULL THEN 1 ELSE 0 END) AS is_desired_partner,
    s.SUBSCRIPTION_STATUS AS source_subscription_status,
    dcr.desired_customer_roles AS source_desired_customer_roles,
    COALESCE(dcr.desired_customer_role_count, 0) AS source_desired_customer_role_count,
    dpr.desired_partner_roles AS source_desired_partner_roles,
    COALESCE(dpr.desired_partner_role_count, 0) AS source_desired_partner_role_count
FROM unknown_meeting_roles umr
INNER JOIN OPENFLOW_DB."public"."classification_industry_roles" ir
    ON ir."id" = umr.recipient_role_id
LEFT JOIN OPENFLOW_DB."public"."businesses" b_src
    ON b_src."id" = umr.source_business_id
LEFT JOIN business_desired_roles bdr_cust
    ON bdr_cust."business_id" = umr.source_business_id
    AND bdr_cust."industry_role_id" = umr.recipient_role_id
    AND bdr_cust."display_type" = 2
LEFT JOIN business_desired_roles bdr_part
    ON bdr_part."business_id" = umr.source_business_id
    AND bdr_part."industry_role_id" = umr.recipient_role_id
    AND bdr_part."display_type" = 3
LEFT JOIN ALIGNABLE_REPORTING_V2.MEMBERSHIPS.DIM_MEMBERSHIP_SUBSCRIPTIONS s
    ON s.BUSINESS_ID = umr.source_business_id
    AND s.ACTIVE_AT >= GREATEST('2026-01-20'::DATE, DATEADD(DAY, -90, CURRENT_DATE()))
LEFT JOIN business_desired_customer_roles_agg dcr
    ON dcr."business_id" = umr.source_business_id
LEFT JOIN business_desired_partner_roles_agg dpr
    ON dpr."business_id" = umr.source_business_id
GROUP BY 1, 2, 3, 10, 11, 12, 13, 14
ORDER BY meeting_count DESC
"""

df_unknown_roles_b2c = query(UNKNOWN_ROLE_BREAKDOWN_QUERY.format(filter="'b2c'"))
print(f"Rows: {len(df_unknown_roles_b2c)}")
df_unknown_roles_b2c.head(30)


In [ ]:
# Distribution of desired role counts across businesses in df_unknown_roles_b2c
biz_role_counts = (
    df_unknown_roles_b2c
    .groupby("SOURCE_BUSINESS_ID")[["SOURCE_DESIRED_CUSTOMER_ROLE_COUNT", "SOURCE_DESIRED_PARTNER_ROLE_COUNT"]]
    .first()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, label, color in [
    (axes[0], "SOURCE_DESIRED_CUSTOMER_ROLE_COUNT", "Desired Customer Roles", "#27ae60"),
    (axes[1], "SOURCE_DESIRED_PARTNER_ROLE_COUNT", "Desired Partner Roles", "#2471a3"),
]:
    counts = biz_role_counts[col].value_counts().sort_index()
    counts = counts[counts >= 10]
    ax.bar(counts.index, counts.values, color=color)
    ax.set_xlabel(f"Number of {label.lower()}")
    ax.set_ylabel("Number of businesses")
    ax.set_title(f"Distribution of {label} per Business")
    for x, y in zip(counts.index, counts.values):
        ax.text(x, y + max(counts.values) * 0.01, str(y), ha="center", va="bottom", fontsize=9)

plt.suptitle(f"B2C Businesses in Unknown Meetings (n={len(biz_role_counts):,} unique businesses, bars with <10 hidden)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Fetch plan_tier for source businesses
df_plan_tiers = query(f'''
    SELECT "id" AS BUSINESS_ID, "plan_tier" AS PLAN_TIER
    FROM OPENFLOW_DB."public"."businesses"
    WHERE "id" IN ({id_list})
''')
plan_tier_map = df_plan_tiers.set_index("BUSINESS_ID")["PLAN_TIER"].to_dict()

# Distribution of desired role counts — paying members only (plan_tier > 0)
paying_biz = biz_role_counts[
    biz_role_counts.index.map(lambda bid: (plan_tier_map.get(bid) or 0) > 0)
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, label, color in [
    (axes[0], "SOURCE_DESIRED_CUSTOMER_ROLE_COUNT", "Desired Customer Roles", "#27ae60"),
    (axes[1], "SOURCE_DESIRED_PARTNER_ROLE_COUNT", "Desired Partner Roles", "#2471a3"),
]:
    counts = paying_biz[col].value_counts().sort_index()
    counts = counts[counts >= 10]
    ax.bar(counts.index, counts.values, color=color)
    ax.set_xlabel(f"Number of {label.lower()}")
    ax.set_ylabel("Number of businesses")
    ax.set_title(f"Distribution of {label} per Business")
    for x, y in zip(counts.index, counts.values):
        ax.text(x, y + max(counts.values) * 0.01, str(y), ha="center", va="bottom", fontsize=9)

plt.suptitle(f"Paying B2C Businesses in Unknown Meetings (n={len(paying_biz):,} unique businesses, bars with <10 hidden)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import importlib, sys, os
# helpers.py lives in the parent notebooks/ directory
sys.path.insert(0, os.path.dirname(os.getcwd()))

if 'helpers' in sys.modules:
    importlib.reload(sys.modules['helpers'])
from helpers import Helpers

h = Helpers(cache_dir="cache/relevance")
h.estimate_relevance_cost(df_unknown_roles_b2c)

In [ ]:
df_unknown_roles_b2c_classified = h.classify_relevance_parallel(df_unknown_roles_b2c)
df_unknown_roles_b2c_classified.head(20)

In [ ]:
n_sample = 5
for rtype in ["potential_customer", "potential_partner", "unrelated"]:
    subset = df_unknown_roles_b2c_classified[
        df_unknown_roles_b2c_classified["RELEVANCE_TYPE"] == rtype
    ]
    sample = subset.sample(n=min(n_sample, len(subset)), random_state=42)
    print(f"\n{'='*80}")
    print(f"  {rtype.upper()}  ({len(subset):,} rows total, showing {len(sample)})")
    print(f"{'='*80}")
    for _, row in sample.iterrows():
        print(f"\n  Source: {row['SOURCE_BUSINESS_NAME']} (id={row['SOURCE_BUSINESS_ID']})")
        print(f"  Desired customers: {row['SOURCE_DESIRED_CUSTOMER_ROLES'] or '(none)'}")
        print(f"  Desired partners:  {row['SOURCE_DESIRED_PARTNER_ROLES'] or '(none)'}")
        print(f"  Recipient role:    {row['RECIPIENT_ROLE']}")
        print(f"  Similar to:        {row['MOST_SIMILAR_DESIRED_ROLE'] or '(none)'}")
        print(f"  Reasoning:         {row['RELEVANCE_REASONING']}")
        print(f"  Meetings:          {row['MEETING_COUNT']} (completed={row['COMPLETED_MEETINGS']}, scheduled={row['SCHEDULED_MEETINGS']})")
        print(f"  ---")

In [ ]:
df_c = df_unknown_roles_b2c_classified
rel_order = ["potential_customer", "potential_partner", "unrelated"]
rel_colors = {"potential_customer": "#27ae60", "potential_partner": "#2471a3", "unrelated": "#c0392b"}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1a. Bar chart by row count
counts = df_c["RELEVANCE_TYPE"].value_counts().reindex(rel_order, fill_value=0)
total = counts.sum()
bars = axes[0].bar(rel_order, counts, color=[rel_colors[r] for r in rel_order])
axes[0].set_title("Unknown Meetings by Relevance Type (row count)")
axes[0].set_ylabel("Number of rows")
for bar, val in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + total*0.01,
                 f"{val:,}\n({val/total:.0%})", ha="center", va="bottom", fontsize=10)

# 1b. Bar chart weighted by MEETING_COUNT
weighted = df_c.groupby("RELEVANCE_TYPE")["MEETING_COUNT"].sum().reindex(rel_order, fill_value=0)
total_w = weighted.sum()
bars = axes[1].bar(rel_order, weighted, color=[rel_colors[r] for r in rel_order])
axes[1].set_title("Unknown Meetings by Relevance Type (weighted by meeting count)")
axes[1].set_ylabel("Total meetings")
for bar, val in zip(bars, weighted):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + total_w*0.01,
                 f"{val:,}\n({val/total_w:.0%})", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Missed roles: LLM says relevant but business didn't list the role as desired
missed = df_c[
    (df_c["RELEVANCE_TYPE"].isin(["potential_customer", "potential_partner"]))
    & (df_c["IS_DESIRED_CUSTOMER"] == 0)
    & (df_c["IS_DESIRED_PARTNER"] == 0)
]
missed_by_role = (
    missed.groupby(["RECIPIENT_ROLE", "RELEVANCE_TYPE"])
    .agg(row_count=("MEETING_COUNT", "size"), total_meetings=("MEETING_COUNT", "sum"))
    .reset_index()
    .sort_values("total_meetings", ascending=False)
)

top_n = 20
top_missed = missed_by_role.head(top_n)

fig, ax = plt.subplots(figsize=(12, 7))
bar_colors = [rel_colors[r] for r in top_missed["RELEVANCE_TYPE"]]
bars = ax.barh(range(len(top_missed)), top_missed["total_meetings"], color=bar_colors)
ax.set_yticks(range(len(top_missed)))
ax.set_yticklabels(top_missed["RECIPIENT_ROLE"], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("Total meetings")
ax.set_title(f"Top {top_n} 'Missed' Roles — LLM says relevant but not in desired roles")
for bar, (_, row) in zip(bars, top_missed.iterrows()):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f"{row['total_meetings']:,} ({row['row_count']} businesses)",
            va="center", fontsize=8)

# Legend
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color=rel_colors["potential_customer"], label="potential_customer"),
    Patch(color=rel_colors["potential_partner"], label="potential_partner"),
], loc="lower right")
plt.tight_layout()
plt.show()

print(f"\nTotal missed role-business combinations: {len(missed):,}")
print(f"Unique missed roles: {missed['RECIPIENT_ROLE'].nunique()}")
print(missed_by_role.head(top_n).to_string(index=False))

In [ ]:
# Fetch business descriptions for all source businesses in the classified df
source_ids = df_c["SOURCE_BUSINESS_ID"].unique().tolist()
id_list = ",".join(str(int(i)) for i in source_ids)
df_descriptions = query(f'''
    SELECT "id" AS BUSINESS_ID, "description" AS BUSINESS_DESCRIPTION
    FROM OPENFLOW_DB."public"."businesses"
    WHERE "id" IN ({id_list})
''')
desc_map = df_descriptions.set_index("BUSINESS_ID")["BUSINESS_DESCRIPTION"].to_dict()

# One example per top missed role
top_missed_roles = missed_by_role.head(top_n)["RECIPIENT_ROLE"].tolist()

for role in top_missed_roles:
    example = missed[missed["RECIPIENT_ROLE"] == role].sample(n=1, random_state=42).iloc[0]
    desc = desc_map.get(example["SOURCE_BUSINESS_ID"], "(no description)")
    print(f"\n{'─'*80}")
    print(f"  Missed role: {role}")
    print(f"  Relevance:   {example['RELEVANCE_TYPE']}")
    print(f"  Source:       {example['SOURCE_BUSINESS_NAME']} (id={example['SOURCE_BUSINESS_ID']})")
    print(f"  Description:  {desc}")
    print(f"  Desired customers: {example['SOURCE_DESIRED_CUSTOMER_ROLES'] or '(none)'}")
    print(f"  Desired partners:  {example['SOURCE_DESIRED_PARTNER_ROLES'] or '(none)'}")
    print(f"  Similar to:        {example['MOST_SIMILAR_DESIRED_ROLE'] or '(none)'}")
    print(f"  Reasoning:         {example['RELEVANCE_REASONING']}")
    print(f"  Meetings:          {example['MEETING_COUNT']} (completed={example['COMPLETED_MEETINGS']}, scheduled={example['SCHEDULED_MEETINGS']})")

In [ ]:
# Scatter: desired customer role count vs proportion of meetings classified as unrelated
biz_stats = df_c.groupby(["SOURCE_BUSINESS_ID", "SOURCE_DESIRED_CUSTOMER_ROLE_COUNT"]).apply(
    lambda g: pd.Series({
        "total_meetings": g["MEETING_COUNT"].sum(),
        "unrelated_meetings": g.loc[g["RELEVANCE_TYPE"] == "unrelated", "MEETING_COUNT"].sum(),
    })
).reset_index()
biz_stats["pct_unrelated"] = biz_stats["unrelated_meetings"] / biz_stats["total_meetings"]
# Only include businesses with enough meetings to be meaningful
biz_stats_filtered = biz_stats[biz_stats["total_meetings"] >= 3]

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(
    biz_stats_filtered["SOURCE_DESIRED_CUSTOMER_ROLE_COUNT"],
    biz_stats_filtered["pct_unrelated"],
    s=biz_stats_filtered["total_meetings"] * 3,
    alpha=0.4,
    edgecolors="white",
    linewidth=0.3,
)
ax.set_xlabel("Number of desired customer roles")
ax.set_ylabel("Proportion of meetings classified as unrelated")
ax.set_title("Desired Customer Role Count vs Unrelated Meeting Proportion (B2C, ≥3 meetings)")
ax.set_ylim(-0.05, 1.05)

# Add trend line
import numpy as np
from numpy.polynomial.polynomial import polyfit
mask = biz_stats_filtered["SOURCE_DESIRED_CUSTOMER_ROLE_COUNT"] >= 0
x = biz_stats_filtered.loc[mask, "SOURCE_DESIRED_CUSTOMER_ROLE_COUNT"]
y = biz_stats_filtered.loc[mask, "pct_unrelated"]
if len(x) > 2:
    b, m = polyfit(x, y, 1)
    x_line = np.array([x.min(), x.max()])
    ax.plot(x_line, b + m * x_line, color="red", linewidth=2, linestyle="--", label=f"trend (slope={m:.3f})")
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Top businesses by total unknown meetings — breakdown by relevance type
biz_totals = (
    df_c.groupby(["SOURCE_BUSINESS_ID", "SOURCE_BUSINESS_NAME"])
    .agg(total_meetings=("MEETING_COUNT", "sum"))
    .reset_index()
    .sort_values("total_meetings", ascending=False)
)
top_businesses = biz_totals.head(15)["SOURCE_BUSINESS_ID"].tolist()

biz_breakdown = (
    df_c[df_c["SOURCE_BUSINESS_ID"].isin(top_businesses)]
    .groupby(["SOURCE_BUSINESS_ID", "SOURCE_BUSINESS_NAME", "RELEVANCE_TYPE"])["MEETING_COUNT"]
    .sum()
    .reset_index()
)
biz_pivot = biz_breakdown.pivot_table(
    index=["SOURCE_BUSINESS_ID", "SOURCE_BUSINESS_NAME"],
    columns="RELEVANCE_TYPE",
    values="MEETING_COUNT",
    fill_value=0,
).reindex(columns=rel_order, fill_value=0)
biz_pivot["total"] = biz_pivot.sum(axis=1)
biz_pivot = biz_pivot.sort_values("total", ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))
y_pos = range(len(biz_pivot))
left = np.zeros(len(biz_pivot))
for rtype in rel_order:
    vals = biz_pivot[rtype].values
    ax.barh(y_pos, vals, left=left, color=rel_colors[rtype], label=rtype)
    left += vals

ax.set_yticks(y_pos)
ax.set_yticklabels([f"{name} ({bid})" for bid, name in biz_pivot.index], fontsize=8)
ax.set_xlabel("Total meetings")
ax.set_title("Top 15 Businesses by Unknown Meetings — Relevance Breakdown")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
# Top recipient roles by relevance type
fig, axes = plt.subplots(1, 3, figsize=(18, 8))
top_n_roles = 15

for ax, rtype in zip(axes, rel_order):
    subset = df_c[df_c["RELEVANCE_TYPE"] == rtype]
    role_counts = (
        subset.groupby("RECIPIENT_ROLE")["MEETING_COUNT"]
        .sum()
        .sort_values(ascending=False)
        .head(top_n_roles)
    )
    role_counts = role_counts.sort_values(ascending=True)
    ax.barh(range(len(role_counts)), role_counts.values, color=rel_colors[rtype])
    ax.set_yticks(range(len(role_counts)))
    ax.set_yticklabels(role_counts.index, fontsize=8)
    ax.set_xlabel("Total meetings")
    ax.set_title(f"Top {top_n_roles} roles: {rtype}")

plt.suptitle("Top Recipient Roles by Relevance Type", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# MOST_SIMILAR_DESIRED_ROLE frequency — "almost matched" roles
similar_counts = (
    df_c[df_c["MOST_SIMILAR_DESIRED_ROLE"].notna() & (df_c["MOST_SIMILAR_DESIRED_ROLE"] != "")]
    .groupby("MOST_SIMILAR_DESIRED_ROLE")
    .agg(row_count=("MEETING_COUNT", "size"), total_meetings=("MEETING_COUNT", "sum"))
    .sort_values("total_meetings", ascending=False)
)

top_n_similar = 20
top_similar = similar_counts.head(top_n_similar)

fig, ax = plt.subplots(figsize=(12, 7))
top_similar_sorted = top_similar.sort_values("total_meetings", ascending=True)
ax.barh(range(len(top_similar_sorted)), top_similar_sorted["total_meetings"], color="#8e44ad")
ax.set_yticks(range(len(top_similar_sorted)))
ax.set_yticklabels(top_similar_sorted.index, fontsize=9)
ax.set_xlabel("Total meetings")
ax.set_title(f"Top {top_n_similar} 'Almost Matched' Desired Roles (MOST_SIMILAR_DESIRED_ROLE)")
for i, (_, row) in enumerate(top_similar_sorted.iterrows()):
    ax.text(row["total_meetings"] + 0.5, i,
            f"{row['total_meetings']:,} ({row['row_count']} combos)",
            va="center", fontsize=8)
plt.tight_layout()
plt.show()

print(similar_counts.head(top_n_similar).to_string())

In [ ]:
churned_pred = df_unknown_b2c_detail["SOURCE_SUBSCRIPTION_STATUS"] == "churned"
activated_pred = df_unknown_b2c_detail["SOURCE_SUBSCRIPTION_STATUS"] == "active"
df_unknown_b2c_detail[churned_pred | activated_pred]

In [ ]:
df_b2b_with_unknown = query(QUERY_WITH_UNKNOWN.format(filter="'b2b'"))
print(df_b2b_with_unknown.to_string(index=False))
plot_with_unknown(df_b2b_with_unknown, "Cancellation Rate for B2B Businesses by Customer/Partner/Unknown × Lifecycle Stage (≥ N Meetings)")

In [ ]:
df_both_split = query(BASE_QUERY_SPLIT_BY_CUSTOMER_PARTNER.format(filter="'both'"))
df_both_split['LIFECYCLE_STAGE'] = df_both_split['LIFECYCLE_STAGE'].str.strip().map(_stage_rename)
print(df_both_split.to_string(index=False))
plot_cancellation_rate(df_both_split, "Cancellation Rate for B2B+B2C Businesses by Customer/Partner × Lifecycle Stage (≥ N Meetings)", min_n=5)

# Restore original globals
lifecycle_order = _orig_lifecycle_order
colors = _orig_colors
linestyles = _orig_linestyles

In [ ]:
USER_COUNT_BY_STAGE_QUERY = """
  WITH conversation_users AS (
      SELECT
          a."conversation_id",
          a."meeting_lifecycle",
          SPLIT_PART(TRANSLATE(c."user_ids", '{{}}"', ''), ',', 1) AS user1,
          SPLIT_PART(TRANSLATE(c."user_ids", '{{}}"', ''), ',', 2) AS user2
      FROM OPENFLOW_DB."public"."conversation_meeting_analyses" a
      INNER JOIN OPENFLOW_DB."public"."conversations" c
          ON c."id" = a."conversation_id"
  ),

  conversation_businesses AS (
      SELECT
          cu."conversation_id",
          cu."meeting_lifecycle",
          u1.BUSINESS_ID AS business1,
          u2.BUSINESS_ID AS business2
      FROM conversation_users cu
      INNER JOIN ALIGNABLE_REPORTING_V2.CORE.DIM_USER u1
          ON u1.ID::VARCHAR = cu.user1 AND u1.IS_PRIMARY_USER = TRUE
      INNER JOIN ALIGNABLE_REPORTING_V2.CORE.DIM_USER u2
          ON u2.ID::VARCHAR = cu.user2 AND u2.IS_PRIMARY_USER = TRUE
  ),

  current_mir_version AS (
      SELECT MAX("version") AS v
      FROM OPENFLOW_DB."public"."classification_mapped_industry_roles"
  ),

  business_desired_roles AS (
      SELECT DISTINCT
          bct."business_id",
          bct."display_type",
          mir."industry_role_id"
      FROM OPENFLOW_DB."public"."classification_business_custom_tags" bct
      CROSS JOIN current_mir_version cmv
      INNER JOIN OPENFLOW_DB."public"."classification_mapped_industry_roles" mir
          ON mir."custom_tag_id" = bct."custom_tag_id"
          AND mir."version" = cmv.v
      WHERE bct."active" = true
        AND bct."display_type" IN (2, 3)
  ),

  business_actual_roles AS (
      SELECT "business_id", "industry_role_id"
      FROM OPENFLOW_DB."public"."classification_business_industry_roles"
  ),

  b1_ideal_flags AS (
      SELECT
          cb."conversation_id",
          MAX(CASE WHEN bdr."display_type" = 2 THEN 1 ELSE 0 END) AS b2_is_customer_of_b1,
          MAX(CASE WHEN bdr."display_type" = 3 THEN 1 ELSE 0 END) AS b2_is_partner_of_b1
      FROM conversation_businesses cb
      INNER JOIN business_actual_roles bar ON bar."business_id" = cb.business2
      INNER JOIN business_desired_roles bdr
          ON bdr."business_id" = cb.business1
          AND bdr."industry_role_id" = bar."industry_role_id"
      GROUP BY 1
  ),

  b2_ideal_flags AS (
      SELECT
          cb."conversation_id",
          MAX(CASE WHEN bdr."display_type" = 2 THEN 1 ELSE 0 END) AS b1_is_customer_of_b2,
          MAX(CASE WHEN bdr."display_type" = 3 THEN 1 ELSE 0 END) AS b1_is_partner_of_b2
      FROM conversation_businesses cb
      INNER JOIN business_actual_roles bar ON bar."business_id" = cb.business1
      INNER JOIN business_desired_roles bdr
          ON bdr."business_id" = cb.business2
          AND bdr."industry_role_id" = bar."industry_role_id"
      GROUP BY 1
  ),

  conversation_ideal_flags AS (
      SELECT
          cb."conversation_id",
          cb."meeting_lifecycle",
          cb.business1,
          cb.business2,
          COALESCE(b1f.b2_is_customer_of_b1, 0) AS b2_is_customer_of_b1,
          COALESCE(b1f.b2_is_partner_of_b1, 0) AS b2_is_partner_of_b1,
          COALESCE(b2f.b1_is_customer_of_b2, 0) AS b1_is_customer_of_b2,
          COALESCE(b2f.b1_is_partner_of_b2, 0) AS b1_is_partner_of_b2
      FROM conversation_businesses cb
      LEFT JOIN b1_ideal_flags b1f ON b1f."conversation_id" = cb."conversation_id"
      LEFT JOIN b2_ideal_flags b2f ON b2f."conversation_id" = cb."conversation_id"
  ),

  meeting_relationships AS (
      SELECT business1 AS business_id, "meeting_lifecycle", 'with_customer' AS relationship_type
      FROM conversation_ideal_flags WHERE b2_is_customer_of_b1 = 1
      UNION ALL
      SELECT business1, "meeting_lifecycle", 'with_partner'
      FROM conversation_ideal_flags WHERE b2_is_partner_of_b1 = 1
      UNION ALL
      SELECT business2, "meeting_lifecycle", 'with_customer'
      FROM conversation_ideal_flags WHERE b1_is_customer_of_b2 = 1
      UNION ALL
      SELECT business2, "meeting_lifecycle", 'with_partner'
      FROM conversation_ideal_flags WHERE b1_is_partner_of_b2 = 1
      UNION ALL
      SELECT business1, "meeting_lifecycle", 'with_unknown'
      FROM conversation_ideal_flags WHERE b2_is_customer_of_b1 = 0 AND b2_is_partner_of_b1 = 0
      UNION ALL
      SELECT business2, "meeting_lifecycle", 'with_unknown'
      FROM conversation_ideal_flags WHERE b1_is_customer_of_b2 = 0 AND b1_is_partner_of_b2 = 0
  ),

  business_stage_counts AS (
      SELECT
          business_id AS BUSINESS_ID,
          "meeting_lifecycle" AS MEETING_LIFECYCLE,
          relationship_type,
          COUNT(*) AS meetings_at_stage
      FROM meeting_relationships
      GROUP BY 1, 2, 3
  ),

  business_customer_type AS (
      SELECT
          b.BUSINESS_ID,
          CASE
              WHEN rcm."customer_type" = 1 THEN 'b2b'
              WHEN rcm."customer_type" = 0 THEN 'b2c'
              WHEN rcm."customer_type" = 2 THEN 'both'
              WHEN rcm."customer_type" = 3 THEN 'none'
              WHEN bf."customer_type" = 0 THEN 'b2b'
              WHEN bf."customer_type" = 1 THEN 'b2c'
              ELSE 'unknown'
          END AS business_type
      FROM (SELECT DISTINCT BUSINESS_ID FROM business_stage_counts) b
      LEFT JOIN OPENFLOW_DB."public"."relationships_business_customer_metadata" rcm
          ON rcm."business_id" = b.BUSINESS_ID
      LEFT JOIN OPENFLOW_DB."public"."business_flags" bf
          ON bf."business_id" = b.BUSINESS_ID
      QUALIFY ROW_NUMBER() OVER (PARTITION BY b.BUSINESS_ID ORDER BY rcm."customer_type" NULLS LAST) = 1
  ),

  thresholds AS (
      SELECT column1 AS threshold FROM VALUES (1),(2),(3),(4),(5),(6),(7),(8),(9),(10)
  ),

  cumulative AS (
      SELECT
          b.MEETING_LIFECYCLE,
          b.relationship_type,
          bct.business_type,
          t.threshold,
          b.BUSINESS_ID
      FROM business_stage_counts b
      CROSS JOIN thresholds t
      INNER JOIN business_customer_type bct ON bct.BUSINESS_ID = b.BUSINESS_ID
      WHERE b.meetings_at_stage >= t.threshold
        AND bct.business_type IN ('b2b', 'b2c', 'both')
  )

  SELECT
      CASE c.MEETING_LIFECYCLE
          WHEN 6 THEN 'completed'
          WHEN 5 THEN 'scheduled'
          WHEN 4 THEN 'mutual_intent'
      END AS lifecycle_stage,
      c.MEETING_LIFECYCLE AS lifecycle_sort,
      c.relationship_type,
      c.business_type,
      c.threshold,
      COUNT(DISTINCT c.BUSINESS_ID) AS num_businesses
  FROM cumulative c
  WHERE c.MEETING_LIFECYCLE IN (4, 5, 6)
  GROUP BY 1, 2, 3, 4, 5
  ORDER BY business_type, lifecycle_sort DESC, relationship_type, threshold ASC
"""

df_user_counts = query(USER_COUNT_BY_STAGE_QUERY)
print(f"Rows: {len(df_user_counts)}")
print(df_user_counts.head(20).to_string(index=False))

In [ ]:
completed_pred = df_user_counts["LIFECYCLE_STAGE"] == "completed"
b2c_pred = df_user_counts["BUSINESS_TYPE"] == "b2c"
met_with_customer_pred = df_user_counts["RELATIONSHIP_TYPE"] == "with_customer"
at_least_one_pred = df_user_counts["THRESHOLD"] == 1
df_user_counts[completed_pred & b2c_pred & met_with_customer_pred & at_least_one_pred]

In [ ]:
user_count_line_styles = {
    ('completed', 'with_customer'):  {'color': '#27ae60', 'linestyle': '-'},
    ('completed', 'with_partner'):   {'color': '#27ae60', 'linestyle': '--'},
    ('completed', 'with_unknown'):   {'color': '#27ae60', 'linestyle': ':'},
    ('scheduled', 'with_customer'):  {'color': '#2471a3', 'linestyle': '-'},
    ('scheduled', 'with_partner'):   {'color': '#2471a3', 'linestyle': '--'},
    ('scheduled', 'with_unknown'):   {'color': '#2471a3', 'linestyle': ':'},
    ('mutual_intent', 'with_customer'): {'color': '#7d3c98', 'linestyle': '-'},
    ('mutual_intent', 'with_partner'):  {'color': '#7d3c98', 'linestyle': '--'},
    ('mutual_intent', 'with_unknown'):  {'color': '#7d3c98', 'linestyle': ':'},
}

biz_type_labels = {'b2b': 'B2B', 'b2c': 'B2C', 'both': 'B2B+B2C'}

for biz_type in ['b2b', 'b2c', 'both']:
    df_biz = df_user_counts[df_user_counts['BUSINESS_TYPE'] == biz_type].copy()
    if df_biz.empty:
        print(f"No data for {biz_type}")
        continue

    df_biz['THRESHOLD'] = df_biz['THRESHOLD'].astype(int)
    df_biz['NUM_BUSINESSES'] = df_biz['NUM_BUSINESSES'].astype(int)

    fig, ax = plt.subplots(figsize=(12, 7))
    texts = []

    for (stage, rel_type), style in user_count_line_styles.items():
        subset = df_biz[
            (df_biz['LIFECYCLE_STAGE'] == stage) &
            (df_biz['RELATIONSHIP_TYPE'] == rel_type)
        ].sort_values('THRESHOLD')
        if subset.empty:
            continue
        label = f"{stage} {rel_type.replace('_', ' ')}"
        ax.plot(
            subset['THRESHOLD'], subset['NUM_BUSINESSES'],
            marker='o', linewidth=2.2, markersize=7,
            label=label, color=style['color'], linestyle=style['linestyle'],
        )
        for _, row in subset.iterrows():
            texts.append(ax.text(
                row['THRESHOLD'], row['NUM_BUSINESSES'],
                f"{int(row['NUM_BUSINESSES']):,}",
                fontsize=7.5, color='#555',
            ))

    ax.set_xticks(range(1, 11))
    ax.set_xticklabels([f'{i}+' for i in range(1, 11)])
    ax.set_xlabel("Minimum # of Meetings at Stage", fontsize=12, labelpad=10)
    ax.set_ylabel("# of Businesses", fontsize=12, labelpad=10)
    ax.set_title(
        f"Number of {biz_type_labels[biz_type]} Businesses with ≥ N Meetings\nby Lifecycle Stage × Customer/Partner/Unknown",
        fontsize=13, pad=15, fontweight='bold',
    )
    ax.legend(title="Stage + Relationship", loc='upper right', fontsize=9, title_fontsize=10, handlelength=3.5)
    ax.grid(axis='y', alpha=0.3)
    sns.despine()
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='#aaa', lw=0.5))
    plt.tight_layout()
    plt.show()

In [ ]:
BUSINESS_IDS_BY_STAGE_QUERY = """
  WITH conversation_users AS (
      SELECT
          a."conversation_id",
          a."meeting_lifecycle",
          SPLIT_PART(TRANSLATE(c."user_ids", '{{}}"', ''), ',', 1) AS user1,
          SPLIT_PART(TRANSLATE(c."user_ids", '{{}}"', ''), ',', 2) AS user2
      FROM OPENFLOW_DB."public"."conversation_meeting_analyses" a
      INNER JOIN OPENFLOW_DB."public"."conversations" c
          ON c."id" = a."conversation_id"
  ),

  conversation_businesses AS (
      SELECT
          cu."conversation_id",
          cu."meeting_lifecycle",
          u1.BUSINESS_ID AS business1,
          u2.BUSINESS_ID AS business2
      FROM conversation_users cu
      INNER JOIN ALIGNABLE_REPORTING_V2.CORE.DIM_USER u1
          ON u1.ID::VARCHAR = cu.user1 AND u1.IS_PRIMARY_USER = TRUE
      INNER JOIN ALIGNABLE_REPORTING_V2.CORE.DIM_USER u2
          ON u2.ID::VARCHAR = cu.user2 AND u2.IS_PRIMARY_USER = TRUE
  ),

  current_mir_version AS (
      SELECT MAX("version") AS v
      FROM OPENFLOW_DB."public"."classification_mapped_industry_roles"
  ),

  business_desired_roles AS (
      SELECT DISTINCT
          bct."business_id",
          bct."display_type",
          mir."industry_role_id"
      FROM OPENFLOW_DB."public"."classification_business_custom_tags" bct
      CROSS JOIN current_mir_version cmv
      INNER JOIN OPENFLOW_DB."public"."classification_mapped_industry_roles" mir
          ON mir."custom_tag_id" = bct."custom_tag_id"
          AND mir."version" = cmv.v
      WHERE bct."active" = true
        AND bct."display_type" IN (2, 3)
  ),

  business_actual_roles AS (
      SELECT "business_id", "industry_role_id"
      FROM OPENFLOW_DB."public"."classification_business_industry_roles"
  ),

  b1_ideal_flags AS (
      SELECT
          cb."conversation_id",
          MAX(CASE WHEN bdr."display_type" = 2 THEN 1 ELSE 0 END) AS b2_is_customer_of_b1,
          MAX(CASE WHEN bdr."display_type" = 3 THEN 1 ELSE 0 END) AS b2_is_partner_of_b1
      FROM conversation_businesses cb
      INNER JOIN business_actual_roles bar ON bar."business_id" = cb.business2
      INNER JOIN business_desired_roles bdr
          ON bdr."business_id" = cb.business1
          AND bdr."industry_role_id" = bar."industry_role_id"
      GROUP BY 1
  ),

  b2_ideal_flags AS (
      SELECT
          cb."conversation_id",
          MAX(CASE WHEN bdr."display_type" = 2 THEN 1 ELSE 0 END) AS b1_is_customer_of_b2,
          MAX(CASE WHEN bdr."display_type" = 3 THEN 1 ELSE 0 END) AS b1_is_partner_of_b2
      FROM conversation_businesses cb
      INNER JOIN business_actual_roles bar ON bar."business_id" = cb.business1
      INNER JOIN business_desired_roles bdr
          ON bdr."business_id" = cb.business2
          AND bdr."industry_role_id" = bar."industry_role_id"
      GROUP BY 1
  ),

  conversation_ideal_flags AS (
      SELECT
          cb."conversation_id",
          cb."meeting_lifecycle",
          cb.business1,
          cb.business2,
          COALESCE(b1f.b2_is_customer_of_b1, 0) AS b2_is_customer_of_b1,
          COALESCE(b1f.b2_is_partner_of_b1, 0) AS b2_is_partner_of_b1,
          COALESCE(b2f.b1_is_customer_of_b2, 0) AS b1_is_customer_of_b2,
          COALESCE(b2f.b1_is_partner_of_b2, 0) AS b1_is_partner_of_b2
      FROM conversation_businesses cb
      LEFT JOIN b1_ideal_flags b1f ON b1f."conversation_id" = cb."conversation_id"
      LEFT JOIN b2_ideal_flags b2f ON b2f."conversation_id" = cb."conversation_id"
  ),

  meeting_relationships AS (
      SELECT business1 AS business_id, "meeting_lifecycle", 'with_customer' AS relationship_type
      FROM conversation_ideal_flags WHERE b2_is_customer_of_b1 = 1
      UNION ALL
      SELECT business1, "meeting_lifecycle", 'with_partner'
      FROM conversation_ideal_flags WHERE b2_is_partner_of_b1 = 1
      UNION ALL
      SELECT business2, "meeting_lifecycle", 'with_customer'
      FROM conversation_ideal_flags WHERE b1_is_customer_of_b2 = 1
      UNION ALL
      SELECT business2, "meeting_lifecycle", 'with_partner'
      FROM conversation_ideal_flags WHERE b1_is_partner_of_b2 = 1
  ),

  business_stage_counts AS (
      SELECT
          business_id AS BUSINESS_ID,
          "meeting_lifecycle" AS MEETING_LIFECYCLE,
          relationship_type,
          COUNT(*) AS meetings_at_stage
      FROM meeting_relationships
      GROUP BY 1, 2, 3
  ),

  business_customer_type AS (
      SELECT
          b.BUSINESS_ID,
          CASE
              WHEN rcm."customer_type" = 1 THEN 'b2b'
              WHEN rcm."customer_type" = 0 THEN 'b2c'
              WHEN rcm."customer_type" = 2 THEN 'both'
              WHEN rcm."customer_type" = 3 THEN 'none'
              WHEN bf."customer_type" = 0 THEN 'b2b'
              WHEN bf."customer_type" = 1 THEN 'b2c'
              ELSE 'unknown'
          END AS business_type
      FROM (SELECT DISTINCT BUSINESS_ID FROM business_stage_counts) b
      LEFT JOIN OPENFLOW_DB."public"."relationships_business_customer_metadata" rcm
          ON rcm."business_id" = b.BUSINESS_ID
      LEFT JOIN OPENFLOW_DB."public"."business_flags" bf
          ON bf."business_id" = b.BUSINESS_ID
      QUALIFY ROW_NUMBER() OVER (PARTITION BY b.BUSINESS_ID ORDER BY rcm."customer_type" NULLS LAST) = 1
  )

  SELECT DISTINCT
      bsc.BUSINESS_ID,
      CASE bsc.MEETING_LIFECYCLE
          WHEN 6 THEN 'completed'
          WHEN 5 THEN 'scheduled'
          WHEN 4 THEN 'mutual_intent'
      END AS lifecycle_stage,
      bsc.relationship_type,
      bct.business_type,
      bsc.meetings_at_stage
  FROM business_stage_counts bsc
  INNER JOIN business_customer_type bct ON bct.BUSINESS_ID = bsc.BUSINESS_ID
  WHERE bsc.MEETING_LIFECYCLE IN (4, 5, 6)
    AND bct.business_type IN ('b2b', 'b2c', 'both')
  ORDER BY bct.business_type, lifecycle_stage, bsc.relationship_type, bsc.BUSINESS_ID
"""

df_business_ids = query(BUSINESS_IDS_BY_STAGE_QUERY)
print(f"Total rows: {len(df_business_ids):,}")
print(f"Unique businesses: {df_business_ids['BUSINESS_ID'].nunique():,}")
print(df_business_ids.head(10).to_string(index=False))

In [ ]:
b2c_completed_customer = df_business_ids[
    (df_business_ids["LIFECYCLE_STAGE"] == "completed") &
    (df_business_ids["BUSINESS_TYPE"] == "b2c") &
    (df_business_ids["RELATIONSHIP_TYPE"] == "with_customer") &
    (df_business_ids["MEETINGS_AT_STAGE"] >= 1)
]
len(b2c_completed_customer["BUSINESS_ID"].tolist())

In [ ]:
5777423 in b2c_completed_customer["BUSINESS_ID"].tolist()

In [ ]:
cohort_ids = b2c_completed_customer["BUSINESS_ID"].dropna().astype(int).unique().tolist()
BATCH = 5000

all_rows = []
for start in range(0, len(cohort_ids), BATCH):
    batch = cohort_ids[start:start + BATCH]
    ids_str = ",".join(str(b) for b in batch)
    rows = query(f"""
        SELECT
            COUNT_IF(s.SUBSCRIPTION_STATUS = 'churned') AS churned,
            COUNT_IF(s.SUBSCRIPTION_STATUS = 'active') AS active,
            churned + active AS total,
            ROUND(churned / NULLIF(churned + active, 0) * 100, 2) AS cancellation_rate_pct
        FROM ALIGNABLE_REPORTING_V2.MEMBERSHIPS.DIM_MEMBERSHIP_SUBSCRIPTIONS s
        WHERE s.BUSINESS_ID IN ({ids_str})
          AND s.ACTIVE_AT >= GREATEST('2026-01-20'::DATE, DATEADD(DAY, -90, CURRENT_DATE()))
    """)
    all_rows.append(rows)

df_cohort = pd.concat(all_rows)
churned = int(df_cohort["CHURNED"].sum())
active = int(df_cohort["ACTIVE"].sum())
total = churned + active
cancel_rate = round(churned / total * 100, 2) if total > 0 else 0

print(f"B2C businesses with ≥1 completed meeting with a customer")
print(f"  Businesses in cohort: {len(cohort_ids):,}")
print(f"  Subscription activity in the last 90 days: {total:,} ({active:,} started, {churned:,} churned)")
print(f"  Cancellation rate:    {cancel_rate}%")

In [ ]:
ids_str = ",".join(str(b) for b in cohort_ids)
df_cohort_subs = query(f"""
    SELECT
        s.BUSINESS_ID,
        s.SUBSCRIPTION_STATUS,
        s.ACTIVE_AT,
        s.*
    FROM ALIGNABLE_REPORTING_V2.MEMBERSHIPS.DIM_MEMBERSHIP_SUBSCRIPTIONS s
    WHERE s.BUSINESS_ID IN ({ids_str})
      AND s.ACTIVE_AT >= GREATEST('2026-01-20'::DATE, DATEADD(DAY, -90, CURRENT_DATE()))
""")
print(f"Rows: {len(df_cohort_subs)}")
df_cohort_subs

In [ ]:
query("""
    SELECT
        a."conversation_id",
        a."meeting_lifecycle",
        c."user_ids"
    FROM OPENFLOW_DB."public"."conversation_meeting_analyses" a
    INNER JOIN OPENFLOW_DB."public"."conversations" c
        ON c."id" = a."conversation_id"
    INNER JOIN ALIGNABLE_REPORTING_V2.CORE.DIM_USER u
        ON u.ID::VARCHAR IN (
            SPLIT_PART(TRANSLATE(c."user_ids", '{}"', ''), ',', 1),
            SPLIT_PART(TRANSLATE(c."user_ids", '{}"', ''), ',', 2)
        )
        AND u.IS_PRIMARY_USER = TRUE
        AND u.BUSINESS_ID = 12614
    WHERE a."meeting_lifecycle" = 6
""")

In [ ]:
query("""
    SELECT
        s.BUSINESS_ID,
        s.SUBSCRIPTION_STATUS,
        s.ACTIVE_AT,
        s.*
    FROM ALIGNABLE_REPORTING_V2.MEMBERSHIPS.DIM_MEMBERSHIP_SUBSCRIPTIONS s
    WHERE s.BUSINESS_ID = 17581286
""")

In [ ]:
query("""
    SELECT
        s.BUSINESS_ID,
        s.SUBSCRIPTION_STATUS,
        s.ACTIVE_AT,
        GREATEST('2026-01-20'::DATE, DATEADD(DAY, -90, CURRENT_DATE())) AS date_cutoff,
        s.ACTIVE_AT >= GREATEST('2026-01-20'::DATE, DATEADD(DAY, -90, CURRENT_DATE())) AS passes_date_filter,
        s.SUBSCRIPTION_STATUS IN ('churned', 'active') AS passes_status_filter,
        s.SUBSCRIPTION_STATUS = 'churned' AS would_count_churned,
        s.SUBSCRIPTION_STATUS = 'active' AS would_count_active
    FROM ALIGNABLE_REPORTING_V2.MEMBERSHIPS.DIM_MEMBERSHIP_SUBSCRIPTIONS s
    WHERE s.BUSINESS_ID = 17581286
""")

In [ ]:
# Debug: trace business 5777423 through each stage of the pipeline

# 1. Does this business have a primary user?
print("=== 1. Primary user lookup ===")
df_debug_user = query("""
    SELECT ID, BUSINESS_ID, IS_PRIMARY_USER
    FROM ALIGNABLE_REPORTING_V2.CORE.DIM_USER
    WHERE BUSINESS_ID = 5777423 AND IS_PRIMARY_USER = TRUE
""")
print(df_debug_user.to_string(index=False))
user_ids = df_debug_user["ID"].tolist()
print(f"User IDs: {user_ids}")

# 2. Does this user appear in any conversation_meeting_analyses with lifecycle=6?
print("\n=== 2. Completed meetings (lifecycle=6) in conversation_meeting_analyses ===")
if user_ids:
    uid_str = ",".join(f"'{int(u)}'" for u in user_ids)
    df_debug_meetings = query(f"""
        SELECT
            a."conversation_id",
            a."meeting_lifecycle",
            c."user_ids"
        FROM OPENFLOW_DB."public"."conversation_meeting_analyses" a
        INNER JOIN OPENFLOW_DB."public"."conversations" c
            ON c."id" = a."conversation_id"
        WHERE a."meeting_lifecycle" = 6
          AND (
            SPLIT_PART(TRANSLATE(c."user_ids", '{{}}\\"', ''), ',', 1) IN ({uid_str})
            OR SPLIT_PART(TRANSLATE(c."user_ids", '{{}}\\"', ''), ',', 2) IN ({uid_str})
          )
    """)
    print(f"Rows: {len(df_debug_meetings)}")
    print(df_debug_meetings.to_string(index=False))
else:
    print("No primary user found — cannot check meetings")

# 3. For those conversations, does the counterpart match as a "customer" via taxonomy?
print("\n=== 3. Counterpart customer/partner classification ===")
if user_ids and len(df_debug_meetings) > 0:
    conv_ids = df_debug_meetings["conversation_id"].tolist()
    conv_str = ",".join(str(int(c)) for c in conv_ids)
    df_debug_flags = query(f"""
        WITH conversation_businesses AS (
            SELECT
                c."id" AS conversation_id,
                u1.BUSINESS_ID AS business1,
                u2.BUSINESS_ID AS business2
            FROM OPENFLOW_DB."public"."conversations" c
            INNER JOIN ALIGNABLE_REPORTING_V2.CORE.DIM_USER u1
                ON u1.ID::VARCHAR = SPLIT_PART(TRANSLATE(c."user_ids", '{{}}\\"', ''), ',', 1)
                AND u1.IS_PRIMARY_USER = TRUE
            INNER JOIN ALIGNABLE_REPORTING_V2.CORE.DIM_USER u2
                ON u2.ID::VARCHAR = SPLIT_PART(TRANSLATE(c."user_ids", '{{}}\\"', ''), ',', 2)
                AND u2.IS_PRIMARY_USER = TRUE
            WHERE c."id" IN ({conv_str})
        ),
        current_mir_version AS (
            SELECT MAX("version") AS v
            FROM OPENFLOW_DB."public"."classification_mapped_industry_roles"
        ),
        business_desired_roles AS (
            SELECT DISTINCT bct."business_id", bct."display_type", mir."industry_role_id"
            FROM OPENFLOW_DB."public"."classification_business_custom_tags" bct
            CROSS JOIN current_mir_version cmv
            INNER JOIN OPENFLOW_DB."public"."classification_mapped_industry_roles" mir
                ON mir."custom_tag_id" = bct."custom_tag_id" AND mir."version" = cmv.v
            WHERE bct."active" = true AND bct."display_type" IN (2, 3)
        ),
        business_actual_roles AS (
            SELECT "business_id", "industry_role_id"
            FROM OPENFLOW_DB."public"."classification_business_industry_roles"
        )
        SELECT
            cb.conversation_id,
            cb.business1,
            cb.business2,
            -- From business1's perspective
            MAX(CASE WHEN bdr1."display_type" = 2 AND bar1."business_id" IS NOT NULL THEN 1 ELSE 0 END) AS b2_is_customer_of_b1,
            MAX(CASE WHEN bdr1."display_type" = 3 AND bar1."business_id" IS NOT NULL THEN 1 ELSE 0 END) AS b2_is_partner_of_b1,
            -- From business2's perspective
            MAX(CASE WHEN bdr2."display_type" = 2 AND bar2."business_id" IS NOT NULL THEN 1 ELSE 0 END) AS b1_is_customer_of_b2,
            MAX(CASE WHEN bdr2."display_type" = 3 AND bar2."business_id" IS NOT NULL THEN 1 ELSE 0 END) AS b1_is_partner_of_b2
        FROM conversation_businesses cb
        LEFT JOIN business_actual_roles bar1 ON bar1."business_id" = cb.business2
        LEFT JOIN business_desired_roles bdr1
            ON bdr1."business_id" = cb.business1 AND bdr1."industry_role_id" = bar1."industry_role_id"
        LEFT JOIN business_actual_roles bar2 ON bar2."business_id" = cb.business1
        LEFT JOIN business_desired_roles bdr2
            ON bdr2."business_id" = cb.business2 AND bdr2."industry_role_id" = bar2."industry_role_id"
        GROUP BY 1, 2, 3
    """)
    print(df_debug_flags.to_string(index=False))
    print("\nFor business 5777423 to appear as 'with_customer', one of these must be true:")
    print("  - b2_is_customer_of_b1=1 (if 5777423 is business1)")
    print("  - b1_is_customer_of_b2=1 (if 5777423 is business2)")

# 4. Business type classification
print("\n=== 4. Business type classification ===")
df_debug_btype = query("""
    SELECT
        rcm."business_id" AS rcm_business_id,
        rcm."customer_type" AS rcm_customer_type,
        bf."business_id" AS bf_business_id,
        bf."customer_type" AS bf_customer_type,
        CASE
            WHEN rcm."customer_type" = 1 THEN 'b2b'
            WHEN rcm."customer_type" = 0 THEN 'b2c'
            WHEN rcm."customer_type" = 2 THEN 'both'
            WHEN rcm."customer_type" = 3 THEN 'none'
            WHEN bf."customer_type" = 0 THEN 'b2b'
            WHEN bf."customer_type" = 1 THEN 'b2c'
            ELSE 'unknown'
        END AS resolved_business_type
    FROM (SELECT 5777423 AS bid) x
    LEFT JOIN OPENFLOW_DB."public"."relationships_business_customer_metadata" rcm
        ON rcm."business_id" = x.bid
    LEFT JOIN OPENFLOW_DB."public"."business_flags" bf
        ON bf."business_id" = x.bid
""")
print(df_debug_btype.to_string(index=False))

In [ ]:
cohort_ids_str = ",".join(str(b) for b in cohort_ids)
df_sub_breakdown = query(f"""
    SELECT
        CASE
            WHEN s.BUSINESS_ID IS NULL THEN 'no_subscription'
            WHEN s.ACTIVE_AT < GREATEST('2026-01-20'::DATE, DATEADD(DAY, -90, CURRENT_DATE())) THEN 'subscription_outside_date_window'
            ELSE 'subscription_in_date_window (' || s.SUBSCRIPTION_STATUS || ')'
        END AS category,
        COUNT(DISTINCT cohort.bid) AS num_businesses
    FROM (SELECT column1 AS bid FROM VALUES {','.join(f'({b})' for b in cohort_ids)}) cohort
    LEFT JOIN ALIGNABLE_REPORTING_V2.MEMBERSHIPS.DIM_MEMBERSHIP_SUBSCRIPTIONS s
        ON s.BUSINESS_ID = cohort.bid
    GROUP BY 1
    ORDER BY 2 DESC
""")
print(df_sub_breakdown.to_string(index=False))

In [ ]:
df_no_sub = query(f"""
    SELECT cohort.bid AS business_id
    FROM (SELECT column1 AS bid FROM VALUES {','.join(f'({b})' for b in cohort_ids)}) cohort
    LEFT JOIN ALIGNABLE_REPORTING_V2.MEMBERSHIPS.DIM_MEMBERSHIP_SUBSCRIPTIONS s
        ON s.BUSINESS_ID = cohort.bid
    WHERE s.BUSINESS_ID IS NULL
    ORDER BY 1
""")
print(f"Businesses with no subscription: {len(df_no_sub)}")
df_no_sub

In [ ]:
df_counts = query("""
SELECT
    CASE
        WHEN rcm."customer_type" = 1 THEN 'b2b'
        WHEN rcm."customer_type" = 0 THEN 'b2c'
        WHEN rcm."customer_type" = 2 THEN 'both'
        WHEN rcm."customer_type" = 3 THEN 'none'
        WHEN bf."customer_type" = 0 THEN 'b2b'
        WHEN bf."customer_type" = 1 THEN 'b2c'
        ELSE 'unknown'
    END AS business_type,
    COUNT(*) AS business_count
FROM ALIGNABLE_REPORTING.BUSINESSES.BUSINESSES b
LEFT JOIN OPENFLOW_DB."public"."relationships_business_customer_metadata" rcm
    ON rcm."business_id" = b.ID
LEFT JOIN OPENFLOW_DB."public"."business_flags" bf
    ON bf."business_id" = b.ID
WHERE b.CREATED_AT >= DATEADD(DAY, -90, CURRENT_DATE())
GROUP BY 1
ORDER BY 2 DESC
""")

print(df_counts.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

bar_colors = {
    'b2b': '#3498db',
    'b2c': '#e74c3c',
    'both': '#9b59b6',
    'none': '#95a5a6',
    'unknown': '#bdc3c7',
}

df_plot = df_counts.sort_values('BUSINESS_COUNT', ascending=True)
ax.barh(
    df_plot['BUSINESS_TYPE'],
    df_plot['BUSINESS_COUNT'],
    color=[bar_colors.get(t, '#333') for t in df_plot['BUSINESS_TYPE']],
)

for i, (_, row) in enumerate(df_plot.iterrows()):
    ax.text(
        row['BUSINESS_COUNT'] + max(df_plot['BUSINESS_COUNT']) * 0.01,
        i,
        f"{int(row['BUSINESS_COUNT']):,}",
        va='center', fontsize=10,
    )

ax.set_xlabel("Number of Businesses", fontsize=12, labelpad=10)
ax.set_title(
    "Businesses Created in Last 90 Days by Customer Type",
    fontsize=13, pad=15, fontweight='bold',
)
ax.grid(axis='x', alpha=0.3)
sns.despine()
plt.tight_layout()
plt.show()

### Deduplicated: Each User in Their Highest Lifecycle Bucket Only

In [ ]:
# Re-query with deduplication: each user gets their MAX lifecycle only
df_lifecycle_churn_deduped = query("""
WITH meeting_users AS (
    SELECT
        MEETING_INITIATOR_USER_ID AS USER_ID,
        MAX(MEETING_LIFECYCLE) AS MEETING_LIFECYCLE
    FROM ALIGNABLE_REPORTING.CONVERSATIONS.CONVERSATION_MEETING_ANALYSES
    GROUP BY 1
),
meeting_businesses AS (
    SELECT DISTINCT u.BUSINESS_ID, mu.MEETING_LIFECYCLE
    FROM ALIGNABLE_REPORTING_V2.CORE.DIM_USER u
    INNER JOIN meeting_users mu ON mu.USER_ID = u.ID::VARCHAR
    WHERE u.IS_PRIMARY_USER = TRUE
),
all_meeting_businesses AS (
    SELECT DISTINCT BUSINESS_ID FROM meeting_businesses
),
lifecycle_stats AS (
    SELECT
        CASE mb.MEETING_LIFECYCLE
            WHEN 6 THEN 'completed (6)'
            WHEN 5 THEN 'scheduled (5)'
            WHEN 4 THEN 'mutual_intent (4)'
            WHEN 3 THEN 'not_interested (3)'
            WHEN 2 THEN 'proposed_didnt_work (2)'
            WHEN 1 THEN 'proposed (1)'
        END AS meeting_lifecycle_filter,
        mb.MEETING_LIFECYCLE AS sort_order,
        COUNT_IF(s.SUBSCRIPTION_STATUS = 'churned') AS churned_subscriptions,
        COUNT_IF(s.SUBSCRIPTION_STATUS = 'active') AS active_subscriptions
    FROM ALIGNABLE_REPORTING_V2.MEMBERSHIPS.DIM_MEMBERSHIP_SUBSCRIPTIONS s
    INNER JOIN meeting_businesses mb ON mb.BUSINESS_ID = s.BUSINESS_ID
    WHERE mb.MEETING_LIFECYCLE IN (1, 2, 3, 4, 5, 6)
      AND s.ACTIVE_AT >= GREATEST('2026-01-20'::DATE, DATEADD(DAY, -90, CURRENT_DATE()))
    GROUP BY 1, 2
),
no_meeting_stats AS (
    SELECT
        'none (no row)' AS meeting_lifecycle_filter,
        -1 AS sort_order,
        COUNT_IF(s.SUBSCRIPTION_STATUS = 'churned') AS churned_subscriptions,
        COUNT_IF(s.SUBSCRIPTION_STATUS = 'active') AS active_subscriptions
    FROM ALIGNABLE_REPORTING_V2.MEMBERSHIPS.DIM_MEMBERSHIP_SUBSCRIPTIONS s
    LEFT JOIN all_meeting_businesses amb ON amb.BUSINESS_ID = s.BUSINESS_ID
    WHERE amb.BUSINESS_ID IS NULL
      AND s.ACTIVE_AT >= GREATEST('2026-01-20'::DATE, DATEADD(DAY, -90, CURRENT_DATE()))
),
combined AS (
    SELECT * FROM lifecycle_stats
    UNION ALL
    SELECT * FROM no_meeting_stats
)
SELECT
    meeting_lifecycle_filter,
    churned_subscriptions,
    active_subscriptions,
    churned_subscriptions + active_subscriptions AS total_subscriptions,
    ROUND(churned_subscriptions / NULLIF(churned_subscriptions + active_subscriptions, 0) * 100, 2) AS cancellation_rate_pct
FROM combined
ORDER BY sort_order DESC
""")

# Combine with same baseline
df_churn_deduped = pd.concat([
    df_lifecycle_churn_deduped[["MEETING_LIFECYCLE_FILTER", "CANCELLATION_RATE_PCT", "TOTAL_SUBSCRIPTIONS"]],
    baseline_row,
], ignore_index=True)

print(df_churn_deduped.to_string(index=False))

In [ ]:
def plot_churn_by_lifecycle_deduped():
    x = range(len(df_churn_deduped))
    fig, ax = plt.subplots(figsize=(12, 6))

    colors = ["coral" if lbl != "Baseline" else "steelblue" for lbl in df_churn_deduped["MEETING_LIFECYCLE_FILTER"]]
    bars = ax.bar(x, df_churn_deduped["CANCELLATION_RATE_PCT"].astype(float), color=colors, edgecolor="white")

    ax.set_xticks(list(x))
    ax.set_xticklabels(df_churn_deduped["MEETING_LIFECYCLE_FILTER"], rotation=30, ha="right", fontsize=9)
    ax.set_ylabel("Cancellation Rate (%)")
    ax.set_title("Membership Cancellation % by Highest Lifecycle (Deduplicated), Last 90 Days", pad=25)

    y_top = ax.get_ylim()[1]
    for i, (bar, rate, n) in enumerate(zip(bars, df_churn_deduped["CANCELLATION_RATE_PCT"].astype(float), df_churn_deduped["TOTAL_SUBSCRIPTIONS"].astype(int))):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f"{rate:.1f}%", ha="center", va="bottom", fontsize=9, fontweight="bold")
        ax.text(i, y_top * 1.02, f"n={n:,}", ha="center", va="bottom", fontsize=7, color="gray")

    plt.tight_layout()
    plt.subplots_adjust(top=0.90)
    return fig

fig = plot_churn_by_lifecycle_deduped()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.colors as mcolors

df_lifecycle_depth = query("""
WITH user_max AS (
    SELECT
        MEETING_INITIATOR_USER_ID AS USER_ID,
        MAX(MEETING_LIFECYCLE) AS MAX_LIFECYCLE
    FROM ALIGNABLE_REPORTING.CONVERSATIONS.CONVERSATION_MEETING_ANALYSES
    GROUP BY 1
),
user_counts AS (
    SELECT
        a.MEETING_INITIATOR_USER_ID AS USER_ID,
        um.MAX_LIFECYCLE,
        COUNT(*) AS MEETINGS_AT_MAX
    FROM ALIGNABLE_REPORTING.CONVERSATIONS.CONVERSATION_MEETING_ANALYSES a
    INNER JOIN user_max um
        ON um.USER_ID = a.MEETING_INITIATOR_USER_ID
        AND a.MEETING_LIFECYCLE = um.MAX_LIFECYCLE
    GROUP BY 1, 2
),
business_counts AS (
    SELECT DISTINCT
        u.BUSINESS_ID,
        uc.MAX_LIFECYCLE,
        uc.MEETINGS_AT_MAX
    FROM ALIGNABLE_REPORTING_V2.CORE.DIM_USER u
    INNER JOIN user_counts uc ON uc.USER_ID = u.ID::VARCHAR
    WHERE u.IS_PRIMARY_USER = TRUE
),
bucketed AS (
    SELECT
        BUSINESS_ID,
        MAX_LIFECYCLE,
        CASE
            WHEN MEETINGS_AT_MAX = 1 THEN '1'
            WHEN MEETINGS_AT_MAX BETWEEN 2 AND 3 THEN '2-3'
            WHEN MEETINGS_AT_MAX BETWEEN 4 AND 5 THEN '4-5'
            ELSE '6+'
        END AS count_bucket,
        CASE
            WHEN MEETINGS_AT_MAX = 1 THEN 1
            WHEN MEETINGS_AT_MAX BETWEEN 2 AND 3 THEN 2
            WHEN MEETINGS_AT_MAX BETWEEN 4 AND 5 THEN 3
            ELSE 4
        END AS bucket_sort
    FROM business_counts
)
SELECT
    CASE b.MAX_LIFECYCLE
        WHEN 6 THEN 'completed (6)'
        WHEN 5 THEN 'scheduled (5)'
        WHEN 4 THEN 'mutual_intent (4)'
        WHEN 3 THEN 'not_interested (3)'
        WHEN 2 THEN 'proposed_didnt_work (2)'
        WHEN 1 THEN 'proposed (1)'
    END AS lifecycle_stage,
    b.MAX_LIFECYCLE AS lifecycle_sort,
    b.count_bucket,
    b.bucket_sort,
    COUNT_IF(s.SUBSCRIPTION_STATUS = 'churned') AS churned,
    COUNT_IF(s.SUBSCRIPTION_STATUS = 'active') AS active,
    churned + active AS total,
    ROUND(churned / NULLIF(churned + active, 0) * 100, 2) AS cancellation_rate_pct
FROM ALIGNABLE_REPORTING_V2.MEMBERSHIPS.DIM_MEMBERSHIP_SUBSCRIPTIONS s
INNER JOIN bucketed b ON b.BUSINESS_ID = s.BUSINESS_ID
WHERE b.MAX_LIFECYCLE IN (1, 2, 3, 4, 5, 6)
  AND s.ACTIVE_AT >= GREATEST('2026-01-20'::DATE, DATEADD(DAY, -90, CURRENT_DATE()))
GROUP BY 1, 2, 3, 4
ORDER BY lifecycle_sort DESC, bucket_sort ASC
""")

print(df_lifecycle_depth.to_string(index=False))

# Build the heatmap
lifecycle_order = [
    'completed (6)', 'scheduled (5)', 'mutual_intent (4)',
    'not_interested (3)', 'proposed_didnt_work (2)', 'proposed (1)'
]
bucket_order = ['1', '2-3', '4-5', '6+']

df_lifecycle_depth['COUNT_BUCKET'] = df_lifecycle_depth['COUNT_BUCKET'].str.strip()
df_lifecycle_depth['LIFECYCLE_STAGE'] = df_lifecycle_depth['LIFECYCLE_STAGE'].str.strip()

# Pivot into matrices
rate_matrix = df_lifecycle_depth.pivot_table(
    index='LIFECYCLE_STAGE', columns='COUNT_BUCKET',
    values='CANCELLATION_RATE_PCT', aggfunc='first'
)
total_matrix = df_lifecycle_depth.pivot_table(
    index='LIFECYCLE_STAGE', columns='COUNT_BUCKET',
    values='TOTAL', aggfunc='first'
)

# Reindex to enforce ordering
rate_matrix = rate_matrix.astype(float)
total_matrix = total_matrix.astype(float)

rate_matrix = rate_matrix.reindex(index=lifecycle_order, columns=bucket_order)
total_matrix = total_matrix.reindex(index=lifecycle_order, columns=bucket_order)

print("Rate matrix index:", rate_matrix.index.tolist())
print("Rate matrix values:")
print(rate_matrix)

import seaborn as sns

def plot_lifecycle_heatmap():
    # Mask cells with n < 5
    mask = total_matrix < 5
    rate_matrix_masked = rate_matrix.copy()
    rate_matrix_masked[mask] = float('nan')

    fig, ax = plt.subplots(figsize=(10, 7))
    
    annot_matrix = rate_matrix_masked.apply(
        lambda col: [
            f"{r:.1f}%\nn={int(t):,}" if pd.notna(r) and pd.notna(t) and t >= 5 else "—"
            for r, t in zip(col, total_matrix[col.name])
        ]
    )
    
    sns.heatmap(
        rate_matrix_masked, ax=ax, cmap='RdYlGn_r', annot=annot_matrix, fmt='',
        linewidths=1, linecolor='white', cbar_kws={'label': 'Cancellation Rate (%)'}
    )
    
    ax.set_xlabel("# Meetings at Highest Lifecycle Stage", fontsize=11, labelpad=10)
    ax.set_ylabel("Highest Lifecycle Stage Reached", fontsize=11, labelpad=10)
    ax.set_title("Cancellation Rate by Lifecycle Stage × Meeting Count", fontsize=13, pad=15)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
    
    plt.tight_layout()
    return fig

fig = plot_lifecycle_heatmap()
plt.show()

In [ ]:
HEATMAP_QUERY = """
WITH user_max AS (
    SELECT
        MEETING_INITIATOR_USER_ID AS USER_ID,
        MAX(MEETING_LIFECYCLE) AS MAX_LIFECYCLE
    FROM ALIGNABLE_REPORTING.CONVERSATIONS.CONVERSATION_MEETING_ANALYSES
    GROUP BY 1
),
user_counts AS (
    SELECT
        a.MEETING_INITIATOR_USER_ID AS USER_ID,
        um.MAX_LIFECYCLE,
        COUNT(*) AS MEETINGS_AT_MAX
    FROM ALIGNABLE_REPORTING.CONVERSATIONS.CONVERSATION_MEETING_ANALYSES a
    INNER JOIN user_max um
        ON um.USER_ID = a.MEETING_INITIATOR_USER_ID
        AND a.MEETING_LIFECYCLE = um.MAX_LIFECYCLE
    GROUP BY 1, 2
),
business_counts AS (
    SELECT DISTINCT
        u.BUSINESS_ID,
        uc.MAX_LIFECYCLE,
        uc.MEETINGS_AT_MAX
    FROM ALIGNABLE_REPORTING_V2.CORE.DIM_USER u
    INNER JOIN user_counts uc ON uc.USER_ID = u.ID::VARCHAR
    WHERE u.IS_PRIMARY_USER = TRUE
),
business_customer_type AS (
    SELECT
        bc.BUSINESS_ID,
        CASE
            WHEN rcm."customer_type" = 1 THEN 'b2b'
            WHEN rcm."customer_type" = 0 THEN 'b2c'
            WHEN rcm."customer_type" = 2 THEN 'both'
            WHEN rcm."customer_type" = 3 THEN 'none'
            WHEN bf."customer_type" = 0 THEN 'b2b'
            WHEN bf."customer_type" = 1 THEN 'b2c'
            ELSE 'unknown'
        END AS business_type
    FROM business_counts bc
    LEFT JOIN OPENFLOW_DB."public"."relationships_business_customer_metadata" rcm
        ON rcm."business_id" = bc.BUSINESS_ID
    LEFT JOIN OPENFLOW_DB."public"."business_flags" bf
        ON bf."business_id" = bc.BUSINESS_ID
    QUALIFY ROW_NUMBER() OVER (PARTITION BY bc.BUSINESS_ID ORDER BY rcm."customer_type" NULLS LAST) = 1
),
bucketed AS (
    SELECT
        bc.BUSINESS_ID,
        bc.MAX_LIFECYCLE,
        CASE
            WHEN bc.MEETINGS_AT_MAX = 1 THEN '1'
            WHEN bc.MEETINGS_AT_MAX BETWEEN 2 AND 3 THEN '2-3'
            WHEN bc.MEETINGS_AT_MAX BETWEEN 4 AND 5 THEN '4-5'
            ELSE '6+'
        END AS count_bucket,
        CASE
            WHEN bc.MEETINGS_AT_MAX = 1 THEN 1
            WHEN bc.MEETINGS_AT_MAX BETWEEN 2 AND 3 THEN 2
            WHEN bc.MEETINGS_AT_MAX BETWEEN 4 AND 5 THEN 3
            ELSE 4
        END AS bucket_sort
    FROM business_counts bc
    INNER JOIN business_customer_type bct ON bct.BUSINESS_ID = bc.BUSINESS_ID
    WHERE bct.business_type = '{biz_type}'
)
SELECT
    CASE b.MAX_LIFECYCLE
        WHEN 6 THEN 'completed (6)'
        WHEN 5 THEN 'scheduled (5)'
        WHEN 4 THEN 'mutual_intent (4)'
        WHEN 3 THEN 'not_interested (3)'
        WHEN 2 THEN 'proposed_didnt_work (2)'
        WHEN 1 THEN 'proposed (1)'
    END AS lifecycle_stage,
    b.MAX_LIFECYCLE AS lifecycle_sort,
    b.count_bucket,
    b.bucket_sort,
    COUNT_IF(s.SUBSCRIPTION_STATUS = 'churned') AS churned,
    COUNT_IF(s.SUBSCRIPTION_STATUS = 'active') AS active,
    churned + active AS total,
    ROUND(churned / NULLIF(churned + active, 0) * 100, 2) AS cancellation_rate_pct
FROM ALIGNABLE_REPORTING_V2.MEMBERSHIPS.DIM_MEMBERSHIP_SUBSCRIPTIONS s
INNER JOIN bucketed b ON b.BUSINESS_ID = s.BUSINESS_ID
WHERE b.MAX_LIFECYCLE IN (1, 2, 3, 4, 5, 6)
  AND s.ACTIVE_AT >= GREATEST('2026-01-20'::DATE, DATEADD(DAY, -90, CURRENT_DATE()))
GROUP BY 1, 2, 3, 4
ORDER BY lifecycle_sort DESC, bucket_sort ASC
"""

LIFECYCLE_ORDER_HEATMAP = [
    'completed (6)', 'scheduled (5)', 'mutual_intent (4)',
    'not_interested (3)', 'proposed_didnt_work (2)', 'proposed (1)'
]
BUCKET_ORDER = ['1', '2-3', '4-5', '6+']

def query_and_plot_heatmap(biz_type, title_label):
    df = query(HEATMAP_QUERY.format(biz_type=biz_type))
    print(df.to_string(index=False))

    df['COUNT_BUCKET'] = df['COUNT_BUCKET'].str.strip()
    df['LIFECYCLE_STAGE'] = df['LIFECYCLE_STAGE'].str.strip()

    rate_matrix = df.pivot_table(
        index='LIFECYCLE_STAGE', columns='COUNT_BUCKET',
        values='CANCELLATION_RATE_PCT', aggfunc='first'
    ).astype(float).reindex(index=LIFECYCLE_ORDER_HEATMAP, columns=BUCKET_ORDER)

    total_matrix = df.pivot_table(
        index='LIFECYCLE_STAGE', columns='COUNT_BUCKET',
        values='TOTAL', aggfunc='first'
    ).astype(float).reindex(index=LIFECYCLE_ORDER_HEATMAP, columns=BUCKET_ORDER)

    # Mask cells with n < 5
    mask = total_matrix < 5
    rate_matrix_masked = rate_matrix.copy()
    rate_matrix_masked[mask] = float('nan')

    fig, ax = plt.subplots(figsize=(10, 7))

    annot_matrix = rate_matrix_masked.apply(
        lambda col: [
            f"{r:.1f}%\nn={int(t):,}" if pd.notna(r) and pd.notna(t) and t >= 5 else "\u2014"
            for r, t in zip(col, total_matrix[col.name])
        ]
    )

    sns.heatmap(
        rate_matrix_masked, ax=ax, cmap='RdYlGn_r', annot=annot_matrix, fmt='',
        linewidths=1, linecolor='white', cbar_kws={'label': 'Cancellation Rate (%)'}
    )

    ax.set_xlabel("# Meetings at Highest Lifecycle Stage", fontsize=11, labelpad=10)
    ax.set_ylabel("Highest Lifecycle Stage Reached", fontsize=11, labelpad=10)
    ax.set_title(f"Cancellation Rate by Lifecycle Stage \u00d7 Meeting Count ({title_label} Only)", fontsize=13, pad=15)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

    plt.tight_layout()
    plt.show()

In [ ]:
query_and_plot_heatmap("b2b", "B2B")

In [ ]:
query_and_plot_heatmap("b2c", "B2C")

In [ ]:
query_and_plot_heatmap("both", "B2B+B2C")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = query("""
WITH user_stage_counts AS (
    SELECT
        a.MEETING_INITIATOR_USER_ID AS USER_ID,
        a.MEETING_LIFECYCLE,
        COUNT(*) AS meetings_at_stage
    FROM ALIGNABLE_REPORTING.CONVERSATIONS.CONVERSATION_MEETING_ANALYSES a
    GROUP BY 1, 2
),
business_stage_counts AS (
    SELECT DISTINCT
        u.BUSINESS_ID,
        usc.MEETING_LIFECYCLE,
        usc.meetings_at_stage
    FROM ALIGNABLE_REPORTING_V2.CORE.DIM_USER u
    INNER JOIN user_stage_counts usc ON usc.USER_ID = u.ID::VARCHAR
    WHERE u.IS_PRIMARY_USER = TRUE
),
thresholds AS (
    SELECT column1 AS threshold FROM VALUES (1),(2),(3),(4),(5),(6),(7),(8),(9),(10)
),
cumulative AS (
    SELECT
        b.MEETING_LIFECYCLE,
        t.threshold,
        b.BUSINESS_ID
    FROM business_stage_counts b
    CROSS JOIN thresholds t
    WHERE b.meetings_at_stage >= t.threshold
)
SELECT
    CASE c.MEETING_LIFECYCLE
        WHEN 6 THEN 'completed (6)'
        WHEN 5 THEN 'scheduled (5)'
        WHEN 4 THEN 'mutual_intent (4)'
    END AS lifecycle_stage,
    c.MEETING_LIFECYCLE AS lifecycle_sort,
    c.threshold,
    COUNT_IF(s.SUBSCRIPTION_STATUS = 'churned') AS churned,
    COUNT_IF(s.SUBSCRIPTION_STATUS = 'active') AS active,
    churned + active AS total,
    ROUND(churned / NULLIF(churned + active, 0) * 100, 2) AS cancellation_rate_pct
FROM ALIGNABLE_REPORTING_V2.MEMBERSHIPS.DIM_MEMBERSHIP_SUBSCRIPTIONS s
INNER JOIN cumulative c ON c.BUSINESS_ID = s.BUSINESS_ID
WHERE c.MEETING_LIFECYCLE IN (4, 5, 6)
  AND s.ACTIVE_AT >= GREATEST('2026-01-20'::DATE, DATEADD(DAY, -90, CURRENT_DATE()))
GROUP BY 1, 2, 3
ORDER BY lifecycle_sort DESC, threshold ASC
""")

print(df.to_string(index=False))

# ── Clean up ──
df['LIFECYCLE_STAGE'] = df['LIFECYCLE_STAGE'].str.strip()
df['CANCELLATION_RATE_PCT'] = df['CANCELLATION_RATE_PCT'].astype(float)
df['THRESHOLD'] = df['THRESHOLD'].astype(int)
df = df.sort_values(['LIFECYCLE_SORT', 'THRESHOLD'])

lifecycle_order = [
    'completed (6)', 'scheduled (5)', 'mutual_intent (4)'
]

colors = {
    'completed (6)': '#2ecc71',
    'scheduled (5)': '#3498db',
    'mutual_intent (4)': '#9b59b6',
}

# ── Plot ──
fig, ax = plt.subplots(figsize=(12, 7))

for stage in lifecycle_order:
    stage_df = df[df['LIFECYCLE_STAGE'] == stage].copy()
    if stage_df.empty:
        continue
    ax.plot(
        stage_df['THRESHOLD'], stage_df['CANCELLATION_RATE_PCT'],
        marker='o', linewidth=2.2, markersize=7,
        label=stage, color=colors.get(stage, '#333'),
    )
    for _, row in stage_df.iterrows():
        ax.annotate(
            f"n={int(row['TOTAL']):,}",
            (row['THRESHOLD'], row['CANCELLATION_RATE_PCT']),
            textcoords="offset points", xytext=(0, 10),
            fontsize=7.5, ha='center', color='#555',
        )

ax.axhline(y=14.3, color='#888', linestyle=':', linewidth=1.5, label='Average (14.3%)')
ax.set_xticks(range(1, 11))
ax.set_xticklabels([f'{i}+' for i in range(1, 11)])
ax.set_xlabel("Minimum # of Meetings at Stage", fontsize=12, labelpad=10)
ax.set_ylabel("Cancellation Rate (%)", fontsize=12, labelpad=10)
ax.set_title(
    "Cancellation Rate for Businesses with ≥ N Meetings at Each Lifecycle Stage",
    fontsize=13, pad=15, fontweight='bold',
)
ax.legend(title="Lifecycle Stage", loc='upper right', fontsize=9, title_fontsize=10)
ax.grid(axis='y', alpha=0.3)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

df = query("""
WITH user_stage_counts AS (
    SELECT
        a.MEETING_INITIATOR_USER_ID AS USER_ID,
        COUNT_IF(a.MEETING_LIFECYCLE = 6) AS completed_count,
        COUNT_IF(a.MEETING_LIFECYCLE = 5) AS scheduled_count
    FROM ALIGNABLE_REPORTING.CONVERSATIONS.CONVERSATION_MEETING_ANALYSES a
    GROUP BY 1
),
business_stage_counts AS (
    SELECT DISTINCT
        u.BUSINESS_ID,
        usc.completed_count,
        usc.scheduled_count
    FROM ALIGNABLE_REPORTING_V2.CORE.DIM_USER u
    INNER JOIN user_stage_counts usc ON usc.USER_ID = u.ID::VARCHAR
    WHERE u.IS_PRIMARY_USER = TRUE
),
bucketed AS (
    SELECT
        BUSINESS_ID,
        completed_count,
        scheduled_count,
        CASE
            WHEN completed_count = 0 THEN '0'
            WHEN completed_count = 1 THEN '1'
            WHEN completed_count = 2 THEN '2'
            WHEN completed_count BETWEEN 3 AND 4 THEN '3-4'
            ELSE '5+'
        END AS completed_bucket,
        CASE
            WHEN completed_count = 0 THEN 0
            WHEN completed_count = 1 THEN 1
            WHEN completed_count = 2 THEN 2
            WHEN completed_count BETWEEN 3 AND 4 THEN 3
            ELSE 4
        END AS bucket_sort
    FROM business_stage_counts
)
SELECT
    b.completed_bucket,
    b.bucket_sort,
    s.SUBSCRIPTION_STATUS,
    COUNT(*) AS num_businesses,
    AVG(b.scheduled_count) AS avg_scheduled,
    MEDIAN(b.scheduled_count) AS median_scheduled
FROM ALIGNABLE_REPORTING_V2.MEMBERSHIPS.DIM_MEMBERSHIP_SUBSCRIPTIONS s
INNER JOIN bucketed b ON b.BUSINESS_ID = s.BUSINESS_ID
WHERE s.SUBSCRIPTION_STATUS IN ('active', 'churned')
  AND s.ACTIVE_AT >= GREATEST('2026-01-20'::DATE, DATEADD(DAY, -90, CURRENT_DATE()))
GROUP BY 1, 2, 3
ORDER BY bucket_sort ASC, SUBSCRIPTION_STATUS
""")

print(df.to_string(index=False))

# ── Clean up ──
df['COMPLETED_BUCKET'] = df['COMPLETED_BUCKET'].str.strip()
df['SUBSCRIPTION_STATUS'] = df['SUBSCRIPTION_STATUS'].str.strip()
df['AVG_SCHEDULED'] = df['AVG_SCHEDULED'].astype(float)
df['BUCKET_SORT'] = df['BUCKET_SORT'].astype(int)
df = df.sort_values(['BUCKET_SORT', 'SUBSCRIPTION_STATUS'])

bucket_order = ['0', '1', '2', '3-4', '5+']

colors = {
    'active': '#2ecc71',
    'churned': '#e74c3c',
}

# ── Plot ──
fig, ax = plt.subplots(figsize=(12, 7))

bar_width = 0.35
x = np.arange(len(bucket_order))

for i, status in enumerate(['active', 'churned']):
    status_df = df[df['SUBSCRIPTION_STATUS'] == status].copy()
    status_df['x_pos'] = status_df['COMPLETED_BUCKET'].map({b: idx for idx, b in enumerate(bucket_order)})
    vals = []
    for b in bucket_order:
        row = status_df[status_df['COMPLETED_BUCKET'] == b]
        vals.append(row['AVG_SCHEDULED'].values[0] if len(row) > 0 else 0)

    offset = -bar_width / 2 + i * bar_width
    bars = ax.bar(x + offset, vals, bar_width, label=status, color=colors[status], alpha=0.85)

    # Annotate n= on each bar
    for j, b in enumerate(bucket_order):
        row = status_df[status_df['COMPLETED_BUCKET'] == b]
        if len(row) > 0:
            n = int(row['NUM_BUSINESSES'].values[0])
            ax.annotate(
                f"n={n:,}",
                (x[j] + offset, vals[j]),
                textcoords="offset points", xytext=(0, 5),
                fontsize=7, ha='center', color='#555',
            )

ax.set_xticks(x)
ax.set_xticklabels(bucket_order)
ax.set_xlabel("# of Completed Meetings", fontsize=12, labelpad=10)
ax.set_ylabel("Avg # of Scheduled (Not Completed) Meetings", fontsize=12, labelpad=10)
ax.set_title(
    "Avg Scheduled Meetings by Completed Meeting Count\n(Active vs Churned)",
    fontsize=13, pad=15, fontweight='bold',
)
ax.legend(title="Subscription Status", fontsize=10, title_fontsize=10)
ax.grid(axis='y', alpha=0.3)
sns.despine()
plt.tight_layout()
plt.show()

### Retention by Highest Lifecycle × Sender Business Model × Recipient Type

In [ ]:
# Save / load df_combined to survive kernel crashes
import os, pickle

df_combined_path = "/Users/garrettsmith/notebooks/edit analysis/.llm_cache/df_combined.pkl"

if os.path.exists(df_combined_path) and 'df_combined' not in dir():
    df_combined = pickle.load(open(df_combined_path, "rb"))
    print(f"Loaded df_combined from cache: {len(df_combined):,} rows, {len(df_combined.columns)} columns")
else:
    pickle.dump(df_combined, open(df_combined_path, "wb"))
    print(f"Saved df_combined to {df_combined_path}: {len(df_combined):,} rows, {len(df_combined.columns)} columns")

In [ ]:
# Query 1: Cancellation rate by meeting lifecycle
df_lifecycle_churn = query("""
WITH meeting_users AS (
    SELECT DISTINCT MEETING_INITIATOR_USER_ID AS USER_ID, MEETING_LIFECYCLE
    FROM ALIGNABLE_REPORTING.CONVERSATIONS.CONVERSATION_MEETING_ANALYSES
),
meeting_businesses AS (
    SELECT DISTINCT u.BUSINESS_ID, mu.MEETING_LIFECYCLE
    FROM ALIGNABLE_REPORTING_V2.CORE.DIM_USER u
    INNER JOIN meeting_users mu ON mu.USER_ID = u.ID::VARCHAR
    WHERE u.IS_PRIMARY_USER = TRUE
),
all_meeting_businesses AS (
    SELECT DISTINCT BUSINESS_ID FROM meeting_businesses
),
lifecycle_stats AS (
    SELECT
        CASE mb.MEETING_LIFECYCLE
            WHEN 6 THEN 'completed (6)'
            WHEN 5 THEN 'scheduled (5)'
            WHEN 4 THEN 'mutual_intent (4)'
            WHEN 3 THEN 'not_interested (3)'
            WHEN 2 THEN 'proposed_didnt_work (2)'
            WHEN 1 THEN 'proposed (1)'
        END AS meeting_lifecycle_filter,
        mb.MEETING_LIFECYCLE AS sort_order,
        COUNT_IF(s.SUBSCRIPTION_STATUS = 'churned') AS churned_subscriptions,
        COUNT_IF(s.SUBSCRIPTION_STATUS = 'active') AS active_subscriptions
    FROM ALIGNABLE_REPORTING_V2.MEMBERSHIPS.DIM_MEMBERSHIP_SUBSCRIPTIONS s
    INNER JOIN meeting_businesses mb ON mb.BUSINESS_ID = s.BUSINESS_ID
    WHERE mb.MEETING_LIFECYCLE IN (1, 2, 3, 4, 5, 6)
      AND s.ACTIVE_AT >= GREATEST('2026-01-20'::DATE, DATEADD(DAY, -90, CURRENT_DATE()))
    GROUP BY 1, 2
),
no_meeting_stats AS (
    SELECT
        'none (no row)' AS meeting_lifecycle_filter,
        -1 AS sort_order,
        COUNT_IF(s.SUBSCRIPTION_STATUS = 'churned') AS churned_subscriptions,
        COUNT_IF(s.SUBSCRIPTION_STATUS = 'active') AS active_subscriptions
    FROM ALIGNABLE_REPORTING_V2.MEMBERSHIPS.DIM_MEMBERSHIP_SUBSCRIPTIONS s
    LEFT JOIN all_meeting_businesses amb ON amb.BUSINESS_ID = s.BUSINESS_ID
    WHERE amb.BUSINESS_ID IS NULL
      AND s.ACTIVE_AT >= GREATEST('2026-01-20'::DATE, DATEADD(DAY, -90, CURRENT_DATE()))
),
combined AS (
    SELECT * FROM lifecycle_stats
    UNION ALL
    SELECT * FROM no_meeting_stats
)
SELECT
    meeting_lifecycle_filter,
    churned_subscriptions,
    active_subscriptions,
    churned_subscriptions + active_subscriptions AS total_subscriptions,
    ROUND(churned_subscriptions / NULLIF(churned_subscriptions + active_subscriptions, 0) * 100, 2) AS cancellation_rate_pct
FROM combined
ORDER BY sort_order DESC
""")

# Query 2: Baseline cancellation rate
df_baseline_churn = query("""
SELECT
    churned_subscriptions,
    active_subscriptions,
    churned_subscriptions + active_subscriptions AS total_subscriptions,
    ROUND(churned_subscriptions / NULLIF(churned_subscriptions + active_subscriptions, 0) * 100, 2) AS cancellation_rate_pct
FROM SEMANTIC_VIEW(
    ALIGNABLE_REPORTING_V2.SEMANTIC.SV_MEMBERSHIP_SUBSCRIPTIONS
    METRICS churned_subscriptions, active_subscriptions
    WHERE subscriptions.active_at >= GREATEST('2026-01-20'::DATE, DATEADD(DAY, -90, CURRENT_DATE()))
)
""")

# Combine: add baseline as a row
baseline_row = pd.DataFrame([{
    "MEETING_LIFECYCLE_FILTER": "Baseline",
    "CANCELLATION_RATE_PCT": float(df_baseline_churn["CANCELLATION_RATE_PCT"].iloc[0]),
    "TOTAL_SUBSCRIPTIONS": int(df_baseline_churn["TOTAL_SUBSCRIPTIONS"].iloc[0]),
}])
df_churn = pd.concat([
    df_lifecycle_churn[["MEETING_LIFECYCLE_FILTER", "CANCELLATION_RATE_PCT", "TOTAL_SUBSCRIPTIONS"]],
    baseline_row,
], ignore_index=True)

print(df_churn.to_string(index=False))

In [ ]:
# For each (source_user, recipient_type) combination,
# find the highest meeting lifecycle achieved.
# Then query churn rates for those businesses.

lifecycle_rank = {"completed": 6, "scheduled": 5, "mutual_intent": 4,
                  "not_interested": 3, "proposed_didnt_work": 2, "proposed": 1, "none": 0}

segments_data = []
for recip_type, recip_col in [("Customer", "context_recipient_matches_sender_customer_targets"),
                               ("Partner", "context_recipient_matches_sender_partner_targets")]:
    subset = df_combined[
        (df_combined[recip_col] == True)
        & (df_combined["outcome_meeting_lifecycle"].notna())
    ].copy()
    if len(subset) == 0:
        continue

    # Per source_user: highest lifecycle
    subset["_lc_rank"] = subset["outcome_meeting_lifecycle"].map(lifecycle_rank).fillna(0)
    best = subset.groupby("source_user_id")["_lc_rank"].max().reset_index()
    best.columns = ["source_user_id", "max_lc_rank"]

    for lc_name, lc_val in [("completed", 6), ("scheduled", 5), ("mutual_intent", 4), ("proposed", 1)]:
        user_ids = best[best["max_lc_rank"] == lc_val]["source_user_id"].tolist()
        if user_ids:
            segments_data.append({
                "label": f"{recip_type} | {lc_name}",
                "recip_type": recip_type,
                "lifecycle": lc_name,
                "user_ids": user_ids,
            })

print(f"Built {len(segments_data)} segments")
for s in segments_data:
    print(f"  {s['label']}: {len(s['user_ids']):,} users")

# Query churn rates for each segment's user_ids
BATCH = 5000
segment_results = []

for seg in segments_data:
    user_ids = seg["user_ids"]
    all_rows = []
    for start in range(0, len(user_ids), BATCH):
        batch = user_ids[start:start + BATCH]
        ids_str = ",".join(str(int(u)) for u in batch)
        rows = query(f"""
            SELECT
                COUNT_IF(s.SUBSCRIPTION_STATUS = 'churned') AS churned,
                COUNT_IF(s.SUBSCRIPTION_STATUS = 'active') AS active
            FROM ALIGNABLE_REPORTING_V2.MEMBERSHIPS.DIM_MEMBERSHIP_SUBSCRIPTIONS s
            INNER JOIN ALIGNABLE_REPORTING_V2.CORE.DIM_USER u
                ON u.BUSINESS_ID = s.BUSINESS_ID AND u.IS_PRIMARY_USER = TRUE
            WHERE u.ID IN ({ids_str})
              AND s.ACTIVE_AT >= GREATEST('2026-01-20'::DATE, DATEADD(DAY, -90, CURRENT_DATE()))
        """)
        all_rows.append(rows)

    df_batch = pd.concat(all_rows)
    churned = int(df_batch["CHURNED"].sum())
    active = int(df_batch["ACTIVE"].sum())
    total = churned + active
    cancel_rate = (churned / total * 100) if total > 0 else 0

    segment_results.append({
        "label": seg["label"],
        "recip_type": seg["recip_type"],
        "lifecycle": seg["lifecycle"],
        "n_users": len(user_ids),
        "total_subscriptions": total,
        "cancellation_rate_pct": round(cancel_rate, 2),
    })

df_segment_churn = pd.DataFrame(segment_results)

# Add baseline
baseline_cancel = float(df_baseline_churn["CANCELLATION_RATE_PCT"].iloc[0])
baseline_total = int(df_baseline_churn["TOTAL_SUBSCRIPTIONS"].iloc[0])

print("\n" + df_segment_churn.to_string(index=False))
print(f"\nBaseline cancellation rate: {baseline_cancel:.2f}% (n={baseline_total:,})")

In [ ]:
# Chart: x-axis = Customer / Partner, bars = lifecycle levels
seg_combos = ["Customer", "Partner"]
seg_labels = ["Customer", "Partner"]
lifecycle_order = ["completed", "scheduled", "mutual_intent", "proposed"]
lc_labels = ["Completed", "Scheduled", "Mutual Intent", "Proposed"]
lc_colors = ["steelblue", "coral", "seagreen", "mediumpurple"]

# Debug: verify df_segment_churn has the expected data
print("Segments in df_segment_churn:")
for _, r in df_segment_churn.iterrows():
    print(f"  {r['recip_type']:<10} {r['lifecycle']:<15} cancel={r['cancellation_rate_pct']:.2f}%  n_subs={r['total_subscriptions']}")

fig, ax = plt.subplots(figsize=(10, 7))
n_groups = len(seg_combos)
n_bars = len(lifecycle_order)
width = 0.18
x = list(range(n_groups))

for bar_idx, (lc, lc_lbl, color) in enumerate(zip(lifecycle_order, lc_labels, lc_colors)):
    offset = (bar_idx - (n_bars - 1) / 2) * width
    vals = []
    n_subs = []
    for rt in seg_combos:
        row = df_segment_churn[
            (df_segment_churn["recip_type"] == rt)
            & (df_segment_churn["lifecycle"] == lc)
        ]
        if len(row) > 0 and int(row["total_subscriptions"].iloc[0]) >= 10:
            vals.append(float(row["cancellation_rate_pct"].iloc[0]))
            n_subs.append(int(row["total_subscriptions"].iloc[0]))
        else:
            vals.append(0)
            n_subs.append(0)

    pos = [i + offset for i in x]
    bars = ax.bar(pos, vals, width, label=lc_lbl, color=color)
    for bar, val, ns in zip(bars, vals, n_subs):
        if ns > 0:
            bx = bar.get_x() + bar.get_width() / 2
            by = max(bar.get_height(), 0) + 0.15
            ax.text(bx, by, f"{val:.1f}%\nn={ns:,}", ha="center", va="bottom", fontsize=7, color="black")

# Baseline as a single wide bar at the end
baseline_x = n_groups + 0.3
bar_width_baseline = n_bars * width
ax.bar(baseline_x, baseline_cancel, bar_width_baseline, color="gray", alpha=0.7, label="Baseline")
ax.text(baseline_x, baseline_cancel + 0.15,
        f"{baseline_cancel:.1f}%\nn={baseline_total:,}", ha="center", va="bottom", fontsize=7, fontweight="bold", color="black")

# Dashed line before baseline
ax.axvline(x=n_groups - 0.3, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)

xlabels = seg_labels + ["Baseline"]
ax.set_xticks(x + [baseline_x])
ax.set_xticklabels(xlabels, fontsize=10)
tick_labels = ax.get_xticklabels()
tick_labels[-1].set_fontweight("bold")

ax.set_ylabel("Cancellation Rate (%)")
ax.set_title("Membership Cancellation % by Recipient Type × Highest Lifecycle\n(Last 90 Days, Each User in Highest Bucket Only)", pad=25)
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()
plt.subplots_adjust(top=0.88)
plt.show()

### Retention by Recipient Type × Highest Lifecycle (Refreshed Data)

In [ ]:
# Self-contained: re-query LLM events for fresh data, extract targeting,
# join meeting lifecycle, then compute churn by segment.
# FUNNEL: each bar includes users who reached AT LEAST that lifecycle stage.

# Step 1: Pull all LLM message pairs with relationship_data JSON
df_fresh = query("""
    SELECT
        g.DATA:source_user_id::INTEGER AS source_user_id,
        g.DATA:target_user_id::INTEGER AS target_user_id,
        g.DATA:tracking_id::STRING AS tracking_id,
        g.DATA AS generated_data
    FROM EVENTS.LLM_MESSAGES.LLM_MESSAGE_EVENTS g
    JOIN EVENTS.LLM_MESSAGES.LLM_MESSAGE_EVENTS s
        ON g.DATA:tracking_id = s.DATA:tracking_id
        AND s.DATA:event_name = 'llm_message_sent'
    WHERE g.DATA:event_name = 'llm_message_generated'
""")
print(f"Fresh LLM message pairs: {len(df_fresh):,}")

# Step 2: Extract customer/partner targeting from JSON
import json as _json

def _extract_targeting(data_str):
    try:
        d = _json.loads(data_str) if isinstance(data_str, str) else data_str
        rd = d.get("context", {}).get("state", {}).get("prompt_context", {}).get("input", {}).get("relationship_data", {})
        return (
            rd.get("recipient_matches_sender_customer_targets", False),
            rd.get("recipient_matches_sender_partner_targets", False),
        )
    except (KeyError, TypeError, _json.JSONDecodeError):
        return (False, False)

targeting = df_fresh["GENERATED_DATA"].apply(_extract_targeting)
df_fresh["is_customer"] = targeting.apply(lambda t: t[0])
df_fresh["is_partner"] = targeting.apply(lambda t: t[1])

print(f"Customer-targeted messages: {df_fresh['is_customer'].sum():,}")
print(f"Partner-targeted messages: {df_fresh['is_partner'].sum():,}")

# Step 3: Get meeting lifecycle for each (source, target) pair via conversations
# Build unique user pairs
pairs = df_fresh[["SOURCE_USER_ID", "TARGET_USER_ID"]].drop_duplicates()
print(f"Unique user pairs: {len(pairs):,}")

# Query meeting lifecycle for these pairs via conversations
source_ids = pairs["SOURCE_USER_ID"].dropna().astype(int).unique().tolist()
BATCH = 5000

conv_rows = []
for start in range(0, len(source_ids), BATCH):
    batch = source_ids[start:start + BATCH]
    ids_str = ",".join(str(u) for u in batch)
    batch_result = query(f"""
        WITH user_convs AS (
            SELECT
                c."id" AS conversation_id,
                c."user_ids",
                REGEXP_SUBSTR(c."user_ids", '\\\\d+', 1, 1)::INTEGER AS user1,
                REGEXP_SUBSTR(c."user_ids", '\\\\d+', 1, 2)::INTEGER AS user2
            FROM OPENFLOW_DB."public"."conversations" c
            WHERE c."conversation_type" = 0
              AND (REGEXP_SUBSTR(c."user_ids", '\\\\d+', 1, 1)::INTEGER IN ({ids_str})
                   OR REGEXP_SUBSTR(c."user_ids", '\\\\d+', 1, 2)::INTEGER IN ({ids_str}))
        )
        SELECT
            uc.user1,
            uc.user2,
            COALESCE(
                CASE cma."meeting_lifecycle"
                    WHEN 6 THEN 'completed'
                    WHEN 5 THEN 'scheduled'
                    WHEN 4 THEN 'mutual_intent'
                    WHEN 3 THEN 'not_interested'
                    WHEN 2 THEN 'proposed_didnt_work'
                    WHEN 1 THEN 'proposed'
                    WHEN 0 THEN 'none'
                END,
                'none'
            ) AS meeting_lifecycle
        FROM user_convs uc
        LEFT JOIN OPENFLOW_DB."public"."conversation_meeting_analyses" cma
            ON cma."conversation_id" = uc.conversation_id
    """)
    conv_rows.append(batch_result)
    if start % 20000 == 0:
        print(f"  Queried conversations for {start + len(batch):,} / {len(source_ids):,} source users")

df_conv = pd.concat(conv_rows, ignore_index=True)
print(f"Conversation-meeting rows: {len(df_conv):,}")

# Step 4: Join meeting lifecycle back to message pairs
df_fresh["meeting_lifecycle"] = "none"
conv_lookup = {}
for _, r in df_conv.iterrows():
    if pd.isna(r["USER1"]) or pd.isna(r["USER2"]):
        continue
    u1, u2, lc = int(r["USER1"]), int(r["USER2"]), r["MEETING_LIFECYCLE"]
    key = (min(u1, u2), max(u1, u2))
    existing = conv_lookup.get(key, "none")
    lifecycle_rank = {"completed": 6, "scheduled": 5, "mutual_intent": 4,
                      "not_interested": 3, "proposed_didnt_work": 2, "proposed": 1, "none": 0}
    if lifecycle_rank.get(lc, 0) > lifecycle_rank.get(existing, 0):
        conv_lookup[key] = lc

df_fresh["meeting_lifecycle"] = [
    conv_lookup.get((min(int(s), int(t)), max(int(s), int(t))), "none")
    if pd.notna(s) and pd.notna(t) else "none"
    for s, t in zip(df_fresh["SOURCE_USER_ID"], df_fresh["TARGET_USER_ID"])
]

print(f"\nMeeting lifecycle distribution:")
print(df_fresh["meeting_lifecycle"].value_counts().to_string())

# Step 5: Compute segments
lifecycle_rank = {"completed": 6, "scheduled": 5, "mutual_intent": 4,
                  "not_interested": 3, "proposed_didnt_work": 2, "proposed": 1, "none": 0}

segments_data = []
for recip_type, recip_col in [("Customer", "is_customer"), ("Partner", "is_partner")]:
    subset = df_fresh[
        (df_fresh[recip_col] == True)
        & (df_fresh["meeting_lifecycle"] != "none")
    ].copy()
    if len(subset) == 0:
        continue

    subset["_lc_rank"] = subset["meeting_lifecycle"].map(lifecycle_rank).fillna(0)
    best = subset.groupby("SOURCE_USER_ID")["_lc_rank"].max().reset_index()
    best.columns = ["source_user_id", "max_lc_rank"]

    for lc_name, lc_val in [("completed", 6), ("scheduled", 5), ("mutual_intent", 4), ("proposed", 1)]:
        user_ids = best[best["max_lc_rank"] >= lc_val]["source_user_id"].tolist()
        if user_ids:
            segments_data.append({
                "label": f"{recip_type} | {lc_name}",
                "recip_type": recip_type,
                "lifecycle": lc_name,
                "user_ids": user_ids,
            })

print(f"\nBuilt {len(segments_data)} segments")
for s in segments_data:
    print(f"  {s['label']}: {len(s['user_ids']):,} users")

# Step 6: Query churn rates
segment_results = []
for seg in segments_data:
    user_ids = seg["user_ids"]
    all_rows = []
    for start in range(0, len(user_ids), BATCH):
        batch = user_ids[start:start + BATCH]
        ids_str = ",".join(str(int(u)) for u in batch)
        rows = query(f"""
            SELECT
                COUNT_IF(s.SUBSCRIPTION_STATUS = 'churned') AS churned,
                COUNT_IF(s.SUBSCRIPTION_STATUS = 'active') AS active
            FROM ALIGNABLE_REPORTING_V2.MEMBERSHIPS.DIM_MEMBERSHIP_SUBSCRIPTIONS s
            INNER JOIN ALIGNABLE_REPORTING_V2.CORE.DIM_USER u
                ON u.BUSINESS_ID = s.BUSINESS_ID AND u.IS_PRIMARY_USER = TRUE
            WHERE u.ID IN ({ids_str})
              AND s.ACTIVE_AT >= GREATEST('2026-01-20'::DATE, DATEADD(DAY, -90, CURRENT_DATE()))
        """)
        all_rows.append(rows)

    df_batch = pd.concat(all_rows)
    churned = int(df_batch["CHURNED"].sum())
    active = int(df_batch["ACTIVE"].sum())
    total = churned + active
    cancel_rate = (churned / total * 100) if total > 0 else 0

    segment_results.append({
        "label": seg["label"],
        "recip_type": seg["recip_type"],
        "lifecycle": seg["lifecycle"],
        "n_users": len(user_ids),
        "total_subscriptions": total,
        "cancellation_rate_pct": round(cancel_rate, 2),
    })

df_segment_churn_fresh = pd.DataFrame(segment_results)

baseline_cancel_fresh = float(df_baseline_churn["CANCELLATION_RATE_PCT"].iloc[0])
baseline_total_fresh = int(df_baseline_churn["TOTAL_SUBSCRIPTIONS"].iloc[0])

print("\n" + df_segment_churn_fresh.to_string(index=False))
print(f"\nBaseline cancellation rate: {baseline_cancel_fresh:.2f}% (n={baseline_total_fresh:,})")

In [ ]:
# Chart: Customer vs Partner × Lifecycle (refreshed data)
seg_combos = ["Customer", "Partner"]
seg_labels = ["Customer", "Partner"]
lifecycle_order = ["completed", "scheduled", "mutual_intent", "proposed"]
lc_labels = ["Completed+", "Scheduled+", "Mutual Intent+", "Proposed+"]
lc_colors = ["steelblue", "coral", "seagreen", "mediumpurple"]

fig, ax = plt.subplots(figsize=(10, 7))
n_groups = len(seg_combos)
n_bars = len(lifecycle_order)
width = 0.18
x = list(range(n_groups))

for bar_idx, (lc, lc_lbl, color) in enumerate(zip(lifecycle_order, lc_labels, lc_colors)):
    offset = (bar_idx - (n_bars - 1) / 2) * width
    vals = []
    n_subs = []
    for rt in seg_combos:
        row = df_segment_churn_fresh[
            (df_segment_churn_fresh["recip_type"] == rt)
            & (df_segment_churn_fresh["lifecycle"] == lc)
        ]
        if len(row) > 0 and int(row["total_subscriptions"].iloc[0]) >= 10:
            vals.append(float(row["cancellation_rate_pct"].iloc[0]))
            n_subs.append(int(row["total_subscriptions"].iloc[0]))
        else:
            vals.append(0)
            n_subs.append(0)

    pos = [i + offset for i in x]
    bars = ax.bar(pos, vals, width, label=lc_lbl, color=color)
    for bar, val, ns in zip(bars, vals, n_subs):
        if ns > 0:
            bx = bar.get_x() + bar.get_width() / 2
            by = max(bar.get_height(), 0) + 0.15
            ax.text(bx, by, f"{val:.1f}%\nn={ns:,}", ha="center", va="bottom", fontsize=7, color="black")

# Baseline
baseline_x = n_groups + 0.3
bar_width_baseline = n_bars * width
ax.bar(baseline_x, baseline_cancel_fresh, bar_width_baseline, color="gray", alpha=0.7, label="Baseline")
ax.text(baseline_x, baseline_cancel_fresh + 0.15,
        f"{baseline_cancel_fresh:.1f}%\nn={baseline_total_fresh:,}", ha="center", va="bottom", fontsize=7, fontweight="bold", color="black")

ax.axvline(x=n_groups - 0.3, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)

xlabels = seg_labels + ["Baseline"]
ax.set_xticks(x + [baseline_x])
ax.set_xticklabels(xlabels, fontsize=10)
tick_labels = ax.get_xticklabels()
tick_labels[-1].set_fontweight("bold")

ax.set_ylabel("Cancellation Rate (%)")
ax.set_title("Membership Cancellation % by Recipient Type × Lifecycle Funnel\n(Users Who Reached At Least Each Stage with That Recipient Type)", pad=25)
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()
plt.subplots_adjust(top=0.88)
plt.show()

### Retention by Business Type (B2B/B2C) × Highest Lifecycle

In [ ]:
# Cancellation rate by business type (b2b/b2c/both) × highest meeting lifecycle.
# Uses relationships_business_customer_metadata (newer) with fallback to business_flags (older).
# FUNNEL: each bar includes users who reached AT LEAST that lifecycle stage.

# Step 1: Get distinct source users from LLM message events
df_biz_fresh = query("""
    SELECT DISTINCT
        g.DATA:source_user_id::INTEGER AS source_user_id
    FROM EVENTS.LLM_MESSAGES.LLM_MESSAGE_EVENTS g
    JOIN EVENTS.LLM_MESSAGES.LLM_MESSAGE_EVENTS s
        ON g.DATA:tracking_id = s.DATA:tracking_id
        AND s.DATA:event_name = 'llm_message_sent'
    WHERE g.DATA:event_name = 'llm_message_generated'
""")
print(f"Distinct source users: {len(df_biz_fresh):,}")

# Step 2: Classify business type
# relationships_business_customer_metadata: consumers(0), businesses(1), both(2), none(3)
# business_flags: businesses(0), consumers(1)
source_ids = df_biz_fresh["SOURCE_USER_ID"].dropna().astype(int).unique().tolist()
BATCH = 5000

biz_type_rows = []
for start in range(0, len(source_ids), BATCH):
    batch = source_ids[start:start + BATCH]
    ids_str = ",".join(str(u) for u in batch)
    rows = query(f"""
        SELECT
            u.ID::INTEGER AS user_id,
            u.BUSINESS_ID,
            CASE
                -- Prefer newer table (rcm)
                WHEN rcm."customer_type" = 1 THEN 'b2b'
                WHEN rcm."customer_type" = 0 THEN 'b2c'
                WHEN rcm."customer_type" = 2 THEN 'both'
                WHEN rcm."customer_type" = 3 THEN 'none'
                -- Fall back to older table (bf)
                WHEN bf."customer_type" = 0 THEN 'b2b'
                WHEN bf."customer_type" = 1 THEN 'b2c'
                ELSE 'unknown'
            END AS business_type
        FROM ALIGNABLE_REPORTING_V2.CORE.DIM_USER u
        LEFT JOIN OPENFLOW_DB."public"."relationships_business_customer_metadata" rcm
            ON rcm."business_id" = u.BUSINESS_ID
        LEFT JOIN OPENFLOW_DB."public"."business_flags" bf
            ON bf."business_id" = u.BUSINESS_ID
        WHERE u.ID IN ({ids_str})
          AND u.IS_PRIMARY_USER = TRUE
    """)
    biz_type_rows.append(rows)

df_biz_type = pd.concat(biz_type_rows, ignore_index=True)
print(f"\nBusiness type distribution:")
print(df_biz_type["BUSINESS_TYPE"].value_counts().to_string())

user_biz_type = dict(zip(df_biz_type["USER_ID"].astype(int), df_biz_type["BUSINESS_TYPE"]))

# Step 3: Get highest meeting lifecycle per source user
meeting_rows = []
for start in range(0, len(source_ids), BATCH):
    batch = source_ids[start:start + BATCH]
    ids_str = ",".join(str(u) for u in batch)
    rows = query(f"""
        SELECT
            TRY_CAST(cma.MEETING_INITIATOR_USER_ID AS INTEGER) AS user_id,
            MAX(cma.MEETING_LIFECYCLE) AS max_lifecycle
        FROM ALIGNABLE_REPORTING.CONVERSATIONS.CONVERSATION_MEETING_ANALYSES cma
        WHERE TRY_CAST(cma.MEETING_INITIATOR_USER_ID AS INTEGER) IN ({ids_str})
        GROUP BY 1
    """)
    meeting_rows.append(rows)

df_meetings = pd.concat(meeting_rows, ignore_index=True)
print(f"\nUsers with meeting data: {len(df_meetings):,}")

lifecycle_map = {6: "completed", 5: "scheduled", 4: "mutual_intent",
                 3: "not_interested", 2: "proposed_didnt_work", 1: "proposed", 0: "none"}
df_meetings["lifecycle"] = df_meetings["MAX_LIFECYCLE"].map(lifecycle_map).fillna("none")
df_meetings["business_type"] = df_meetings["USER_ID"].map(user_biz_type).fillna("unknown")

print(f"\nLifecycle × Business Type:")
print(pd.crosstab(df_meetings["business_type"], df_meetings["lifecycle"]).to_string())

# Step 4: Build segments and query churn
segments_data_biz = []
for biz_type in ["b2b", "b2c", "both"]:
    for lc_name, lc_val in [("completed", 6), ("scheduled", 5), ("mutual_intent", 4), ("proposed", 1)]:
        user_ids = df_meetings[
            (df_meetings["business_type"] == biz_type)
            & (df_meetings["MAX_LIFECYCLE"] >= lc_val)
        ]["USER_ID"].tolist()
        if user_ids:
            segments_data_biz.append({
                "label": f"{biz_type.upper()} | {lc_name}",
                "business_type": biz_type,
                "lifecycle": lc_name,
                "user_ids": user_ids,
            })

print(f"\nBuilt {len(segments_data_biz)} segments")
for s in segments_data_biz:
    print(f"  {s['label']}: {len(s['user_ids']):,} users")

segment_results_biz = []
for seg in segments_data_biz:
    user_ids = seg["user_ids"]
    all_rows = []
    for start in range(0, len(user_ids), BATCH):
        batch = user_ids[start:start + BATCH]
        ids_str = ",".join(str(int(u)) for u in batch)
        rows = query(f"""
            SELECT
                COUNT_IF(s.SUBSCRIPTION_STATUS = 'churned') AS churned,
                COUNT_IF(s.SUBSCRIPTION_STATUS = 'active') AS active
            FROM ALIGNABLE_REPORTING_V2.MEMBERSHIPS.DIM_MEMBERSHIP_SUBSCRIPTIONS s
            INNER JOIN ALIGNABLE_REPORTING_V2.CORE.DIM_USER u
                ON u.BUSINESS_ID = s.BUSINESS_ID AND u.IS_PRIMARY_USER = TRUE
            WHERE u.ID IN ({ids_str})
              AND s.ACTIVE_AT >= GREATEST('2026-01-20'::DATE, DATEADD(DAY, -90, CURRENT_DATE()))
        """)
        all_rows.append(rows)

    df_batch = pd.concat(all_rows)
    churned = int(df_batch["CHURNED"].sum())
    active = int(df_batch["ACTIVE"].sum())
    total = churned + active
    cancel_rate = (churned / total * 100) if total > 0 else 0

    segment_results_biz.append({
        "label": seg["label"],
        "business_type": seg["business_type"],
        "lifecycle": seg["lifecycle"],
        "n_users": len(user_ids),
        "total_subscriptions": total,
        "cancellation_rate_pct": round(cancel_rate, 2),
    })

df_segment_churn_biz = pd.DataFrame(segment_results_biz)

baseline_cancel_biz = float(df_baseline_churn["CANCELLATION_RATE_PCT"].iloc[0])
baseline_total_biz = int(df_baseline_churn["TOTAL_SUBSCRIPTIONS"].iloc[0])

print("\n" + df_segment_churn_biz.to_string(index=False))
print(f"\nBaseline cancellation rate: {baseline_cancel_biz:.2f}% (n={baseline_total_biz:,})")

In [ ]:
# Chart: B2B / B2C / Both × Lifecycle
seg_combos = ["b2b", "b2c", "both"]
seg_labels = ["B2B", "B2C", "Both"]
lifecycle_order = ["completed", "scheduled", "mutual_intent", "proposed"]
lc_labels = ["Completed+", "Scheduled+", "Mutual Intent+", "Proposed+"]
lc_colors = ["steelblue", "coral", "seagreen", "mediumpurple"]

fig, ax = plt.subplots(figsize=(12, 7))
n_groups = len(seg_combos)
n_bars = len(lifecycle_order)
width = 0.18
x = list(range(n_groups))

for bar_idx, (lc, lc_lbl, color) in enumerate(zip(lifecycle_order, lc_labels, lc_colors)):
    offset = (bar_idx - (n_bars - 1) / 2) * width
    vals = []
    n_subs = []
    for bt in seg_combos:
        row = df_segment_churn_biz[
            (df_segment_churn_biz["business_type"] == bt)
            & (df_segment_churn_biz["lifecycle"] == lc)
        ]
        if len(row) > 0 and int(row["total_subscriptions"].iloc[0]) >= 1:
            vals.append(float(row["cancellation_rate_pct"].iloc[0]))
            n_subs.append(int(row["total_subscriptions"].iloc[0]))
        else:
            vals.append(0)
            n_subs.append(0)

    pos = [i + offset for i in x]
    bars = ax.bar(pos, vals, width, label=lc_lbl, color=color)
    for bar, val, ns in zip(bars, vals, n_subs):
        if ns > 0:
            bx = bar.get_x() + bar.get_width() / 2
            by = max(bar.get_height(), 0) + 0.15
            ax.text(bx, by, f"{val:.1f}%\nn={ns:,}", ha="center", va="bottom", fontsize=7, color="black")

# Baseline
baseline_x = n_groups + 0.3
bar_width_baseline = n_bars * width
ax.bar(baseline_x, baseline_cancel_biz, bar_width_baseline, color="gray", alpha=0.7, label="Baseline")
ax.text(baseline_x, baseline_cancel_biz + 0.15,
        f"{baseline_cancel_biz:.1f}%\nn={baseline_total_biz:,}", ha="center", va="bottom", fontsize=7, fontweight="bold", color="black")

ax.axvline(x=n_groups - 0.3, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)

xlabels = seg_labels + ["Baseline"]
ax.set_xticks(x + [baseline_x])
ax.set_xticklabels(xlabels, fontsize=10)
tick_labels = ax.get_xticklabels()
tick_labels[-1].set_fontweight("bold")

ax.set_ylabel("Cancellation Rate (%)")
ax.set_title("Membership Cancellation % by Business Type × Meeting Lifecycle Funnel\n(Last 90 Days, Users Who Reached At Least Each Stage)", pad=25)
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()
plt.subplots_adjust(top=0.88)
plt.show()